# ECLIPSE — w/o EC, Training Seed 3

Independent ablation training: **only `evidence_delta=0.0` differs from full ECLIPSE**.
Keep WMSG, CF support expansion, LC, MI/FSC/BPR/BSC and the actual seed-1 dataset hyperparameters.

Seed 1 has native recovered w/o EC receipts; seed 2 has completed independent training on all three datasets.
Both sets of measured receipts are embedded separately with audited provenance for aggregation only.
This notebook runs **SEED=3** from fresh initialization. It never loads seed-1/2 weights for training.

Training: 64D, 500 epochs, VALID NDCG@20 checkpoint selection. Report TEST only from the restored selected checkpoint;
the retained native evaluator may also log terminal epoch-500 TEST, which is excluded from the paper table.
No epoch-based p-values, fabricated missing metrics or full-model receipts inserted into this ablation.

Outputs follow seed 2: PDF vector + PNG 300 dpi, source CSV/JSON, captions, manifests,
selected TEST receipts, checkpoints, telemetry, seed summaries and ZIP. No model tests or training run automatically during setup.

**Kaggle prerequisite:** attach `rainyle/stair-datasets-mmrec` (or matching processed data) using **Add Input**
before Save & Run All. Internet alone cannot attach a new datasource during a batch job.


## 1. Cấu hình phiên chạy — chỉnh tại một nơi


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import sys
import uuid

SEED = 3
SELECTED_DATASETS = ('sports', 'baby', 'electronics')
GPU_INDEX = 0
INSTALL_MISSING_DEPENDENCIES = True
AUTO_DOWNLOAD_DATA = False  # Optional outside Kaggle only; attach inputs before Save & Run All.
DATASET_HANDLE = 'rainyle/stair-datasets-mmrec'
DATASET_SEARCH_ROOTS = []  # Optional parent folders; default input/cache paths are searched automatically.
STRICT_RUNTIME = True  # Require the seed-1 Python major/minor, Torch and FreeRec versions.
RUNTIME_COMMIT = '5212263e87c5c1cfe6e48c91afa90201a48f5929'
REPO_URL = 'https://github.com/ThanhChuong12/STAIR-Enhanced.git'
REPO_OVERRIDE = None  # Optional existing repository; checked against seed-1 source digests.
DATASET_SOURCE_OVERRIDES = {}  # Example: {'electronics': '/kaggle/input/.../electronics'}
ADDITIONAL_ARTIFACT_ROOTS = []  # Optional: additional w/o EC receipts only. Audited seed-1/2 measurements are already embedded.

if isinstance(SEED, bool) or SEED not in (2, 3):
    raise ValueError('This notebook is for independent w/o EC training seeds 2 or 3.')
if not SELECTED_DATASETS or len(set(SELECTED_DATASETS)) != len(SELECTED_DATASETS):
    raise ValueError('Select one or more datasets without duplicates.')
if set(SELECTED_DATASETS) - {'baby', 'sports', 'electronics'}:
    raise ValueError('Unknown dataset key.')
if isinstance(GPU_INDEX, bool) or not isinstance(GPU_INDEX, int) or GPU_INDEX < 0:
    raise ValueError('GPU_INDEX must be a non-negative integer.')

WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
SESSION_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
RUN_ROOT = WORK_DIR / 'eclipse_runs' / f'eclipse_wo_ec_seed{SEED}_{SESSION_ID}'
for name in ('logs', 'artifacts', 'configs', 'figures', 'tables', 'provenance'):
    (RUN_ROOT / name).mkdir(parents=True, exist_ok=True)
DATA_ROOT = WORK_DIR / 'eclipse_data'
GRAPH_CACHE_ROOT = WORK_DIR / 'eclipse_graph_cache' / RUNTIME_COMMIT[:12]
GRAPH_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

PRIMARY_METRICS = ('RECALL@10', 'RECALL@20', 'NDCG@10', 'NDCG@20')
DATASET_TITLES = {'baby': 'Amazon Baby', 'sports': 'Amazon Sports', 'electronics': 'Amazon Electronics'}
DATASET_FOLDERS = {k: f'Amazon2014{v}_550_MMRec' for k, v in
                   [('baby', 'Baby'), ('sports', 'Sports'), ('electronics', 'Electronics')]}
print(f'ECLIPSE w/o EC | seed={SEED} | datasets={SELECTED_DATASETS}')
print(f'Run directory: {RUN_ROOT}')
print('Training starts only in the dataset training cells below.')


## 2. Locked protocol and verified w/o EC seed-1 measurements

`SEED1_REFERENCE` holds full-model **protocol** references; its TEST values never enter w/o EC aggregation.
`WO_EC_GRAPH` holds the exact uncalibrated operator hashes.
`VERIFIED_WO_EC_SEED1` holds separately verified **measured w/o EC** TEST receipts recovered from the original
best checkpoints (Baby 430, Sports/Electronics 500), with receipt/source/checkpoint provenance.
The snapshots are actual seed-1 results, not a new training run. This removes the need to upload old checkpoints again merely to aggregate.

Seed-2 w/o EC snapshots below were checked against supplied native receipts, manifests, source digests and checkpoint byte hashes. They are not full-model seed-2 results. A manifest projection retains the fields required by the aggregation contract; original file digests and canonical snapshot digests are stored separately.


In [ ]:
SEED1_REFERENCE = json.loads('{"baby":{"commit":"5212263e87c5c1cfe6e48c91afa90201a48f5929","python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","gpu":"Tesla T4","v8_config":{"embedding_dim":64,"num_layers":3,"gamma":0.1,"lr":0.001,"weight_decay":0.3,"batch_size":1024,"epochs":500,"beta1":0.9,"beta2":0.999,"mfiles":["textual_modality.pkl","visual_modality.pkl"],"num_neighbors":[5,1],"eval_freq":5,"lambda_nlgcl":0.01,"nlgcl_tau":0.2,"nlgcl_G":1,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"cl_chunk_size":1024,"knn_chunk_size":1024,"knn_device":"cpu","cf_block_size":64,"cf_memory_budget_mib":128.0},"icaart_config":{"evidence_mode":"pooled_activity_excess","evidence_activity_bins":"log2","evidence_halves":0,"evidence_seed":null,"evidence_tau":2.0,"evidence_delta":1.0,"evidence_candidate_block":8192,"evidence_placebo_seed":20261016},"features":[{"file":"textual_modality.pkl","sha256":"ceda3582c04f4e276e8d4bdcc2146f2634f6ca171e1f9dfafeb219dc5cfe96eb"},{"file":"visual_modality.pkl","sha256":"763bf9510c74fd5c6a45ee8164ca9f1ababc5c355c1757fc3acbdcc5d46af0a5"}],"knn":[{"features":"ceda3582c04f4e276e8d4bdcc2146f2634f6ca171e1f9dfafeb219dc5cfe96eb","k":5,"chunk":1024,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"2cb2ad16db4996b65a179bbb221bbf09ac3eb390cf2cacf2cb3b26fd471efa40","cache_hit":false,"candidate_sha256":"5bfeae7c9bb0a96be4ca20803e20d0d84a57068aeb322749bd132f0ce2691827"},{"features":"763bf9510c74fd5c6a45ee8164ca9f1ababc5c355c1757fc3acbdcc5d46af0a5","k":1,"chunk":1024,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"a50d5b1b5a586f38a4a5aec66d128556d43046d899d631a8756b476d47e04586","cache_hit":false,"candidate_sha256":"ce97e02fbb82f94038c40fd2ae8e911c387c951b7710ee29eb2b144fc3cd7b08"}],"contract":{"config":{"embedding_dim":64,"num_layers":3,"gamma":0.1,"num_neighbors":[5,1],"mfiles":["textual_modality.pkl","visual_modality.pkl"],"lr":0.001,"weight_decay":0.3,"beta1":0.9,"beta2":0.999,"batch_size":1024,"seed":1,"eval_freq":5,"ranking":"full","which4best":"NDCG@20","num_workers":4,"nlgcl_G":1,"nlgcl_tau":0.2,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"placebo_seed":1,"knn_device":"cpu","knn_chunk_size":1024,"cl_chunk_size":1024,"metadata_fingerprint":null},"splits":{"test.txt":"307162309a9fae52172dabbbb0f373e845af5e73740d987b70bb0f406d5ac411","train.txt":"bcced50dea427734d76b031f1de140f219d67ba6bb6bc5eac85d7b3be51645bb","valid.txt":"c44687959888d29e2d931aa3de1210f7f708e204d169b7680e5adfcf0a183a3a"},"candidate_hashes":["5bfeae7c9bb0a96be4ca20803e20d0d84a57068aeb322749bd132f0ce2691827","ce97e02fbb82f94038c40fd2ae8e911c387c951b7710ee29eb2b144fc3cd7b08"],"sources":{"stair5_v8.py":"dea612a122c82f7cc6c4c1ba069510fa1242fa3819f00b6cbfe0d37b9ed9c502","stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","freerec_compat.py":"de46f3dd9bc11bb69acd5f151d175968e73d980be37ad36d5afc636b0b9c5e90","stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8","stair_icaart.py":"77b862894f9aa91eba8fa96ad54a38eb6e29a25d9e134b7718a58f4bb5851014","stair_icaart_evidence.py":"11487e0fef87077cde7863b58847bcf22adeebef6e5378ae32f71f366d88c41a","stair_icaart_graph.py":"33bd582c8cd3b298eed1ce47ad99837ec9f0cd220778d684ae3d96161a455aad","stair_icaart_config.py":"fe5bcaa711ba64fd40d431db27ae6278e7c937ce05f57786b6f7262668061a51","main_stair_icaart.py":"b95eb5423ac8947da290bd13562e0aeb433adb8742ca0f0246139b4880aeec3f","stair_icaart_reporting.py":"6902360e9ac92dd511ea87e988ad622cae51886eb0ac183b170ce7df1b5065cc"},"data_fingerprint":"de38b811bc05e4838c2f52d025ee3464447a1af4e1a1844747635cddafd13a9e","graph_fingerprint":"b8013795d439064965dcd6946a44751d9a275ca0cefdc5231192066582d757a8"},"selected_receipt":{"mode":"test","epoch":480,"step":-1,"selected_checkpoint":true,"selection":"validation NDCG@20","seed":1,"evidence_mode":"pooled_activity_excess","evidence_delta":1.0,"graph_fingerprint":"b8013795d439064965dcd6946a44751d9a275ca0cefdc5231192066582d757a8","metrics":{"RECALL@10":0.06861877473361941,"RECALL@20":0.10610715503188693,"NDCG@10":0.03679847170953948,"NDCG@20":0.046397878786352126}},"provenance":"logs/GD6/STAIR-PEEC/stair_icaart/baby_STAIR_ICAART_pooled_activity_excess_d1.0_seed1","fit_seconds":1214.194699},"sports":{"commit":"5212263e87c5c1cfe6e48c91afa90201a48f5929","python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","gpu":"Tesla T4","v8_config":{"embedding_dim":64,"num_layers":3,"gamma":0.2,"lr":0.001,"weight_decay":0.1,"batch_size":1024,"epochs":500,"beta1":0.9,"beta2":0.999,"mfiles":["textual_modality.pkl","visual_modality.pkl"],"num_neighbors":[5,1],"eval_freq":5,"lambda_nlgcl":0.01,"nlgcl_tau":0.2,"nlgcl_G":1,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"cl_chunk_size":1024,"knn_chunk_size":1024,"knn_device":"cpu","cf_block_size":64,"cf_memory_budget_mib":128.0},"icaart_config":{"evidence_mode":"pooled_activity_excess","evidence_activity_bins":"log2","evidence_halves":0,"evidence_seed":null,"evidence_tau":2.0,"evidence_delta":1.0,"evidence_candidate_block":8192,"evidence_placebo_seed":20261016},"features":[{"file":"textual_modality.pkl","sha256":"ca99972f3a2c7a9ead14e9a0d649351411722bbd98e920b4c5fe9f8ffa1564a2"},{"file":"visual_modality.pkl","sha256":"5d2ea8187bbc52f806616404815930061acefe6cb22f353e3f02f286c95b9959"}],"knn":[{"features":"ca99972f3a2c7a9ead14e9a0d649351411722bbd98e920b4c5fe9f8ffa1564a2","k":5,"chunk":609,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"9dacd55f8d2be69d2bebff457c2c83345135e05f90eb44c1c4f0b8d62b4347e7","cache_hit":false,"candidate_sha256":"bbe9e918209a03b5186a74cbd78ba474f420dfaab4eb6c2095c7c578ac0ca5af"},{"features":"5d2ea8187bbc52f806616404815930061acefe6cb22f353e3f02f286c95b9959","k":1,"chunk":609,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"9a875c92e35fbe67a4e43ac1298d898fed91f7160d449e4d9e8d5b86899889e5","cache_hit":false,"candidate_sha256":"e3270692a18ae7d11d50a83d30b7f071f4a095e630714c01b2a0f6b4cff4e8a5"}],"contract":{"config":{"embedding_dim":64,"num_layers":3,"gamma":0.2,"num_neighbors":[5,1],"mfiles":["textual_modality.pkl","visual_modality.pkl"],"lr":0.001,"weight_decay":0.1,"beta1":0.9,"beta2":0.999,"batch_size":1024,"seed":1,"eval_freq":5,"ranking":"full","which4best":"NDCG@20","num_workers":4,"nlgcl_G":1,"nlgcl_tau":0.2,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"placebo_seed":1,"knn_device":"cpu","knn_chunk_size":1024,"cl_chunk_size":1024,"metadata_fingerprint":null},"splits":{"test.txt":"218501613d897422d24d2035e268af027f4ca77688a3f836080b197621f5610a","train.txt":"2491704e8543dbdeb9046cec0928c6734aa80521b1026afd53e0d296f4f7427c","valid.txt":"6dabc7efcc14de50909c6b1995f7bf9862d6233a3440d593d389e1bf91fef56e"},"candidate_hashes":["bbe9e918209a03b5186a74cbd78ba474f420dfaab4eb6c2095c7c578ac0ca5af","e3270692a18ae7d11d50a83d30b7f071f4a095e630714c01b2a0f6b4cff4e8a5"],"sources":{"stair5_v8.py":"dea612a122c82f7cc6c4c1ba069510fa1242fa3819f00b6cbfe0d37b9ed9c502","stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","freerec_compat.py":"de46f3dd9bc11bb69acd5f151d175968e73d980be37ad36d5afc636b0b9c5e90","stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8","stair_icaart.py":"77b862894f9aa91eba8fa96ad54a38eb6e29a25d9e134b7718a58f4bb5851014","stair_icaart_evidence.py":"11487e0fef87077cde7863b58847bcf22adeebef6e5378ae32f71f366d88c41a","stair_icaart_graph.py":"33bd582c8cd3b298eed1ce47ad99837ec9f0cd220778d684ae3d96161a455aad","stair_icaart_config.py":"fe5bcaa711ba64fd40d431db27ae6278e7c937ce05f57786b6f7262668061a51","main_stair_icaart.py":"b95eb5423ac8947da290bd13562e0aeb433adb8742ca0f0246139b4880aeec3f","stair_icaart_reporting.py":"6902360e9ac92dd511ea87e988ad622cae51886eb0ac183b170ce7df1b5065cc"},"data_fingerprint":"383929cee0a814ae8211bde6c15ef0abb6ecd4499822bdf5f3579cb87fa04643","graph_fingerprint":"6b7be789b969cd17a17459deea25bfdf5bfcf518a008b02b7c2925a2d3f88f6c"},"selected_receipt":{"mode":"test","epoch":500,"step":-1,"selected_checkpoint":true,"selection":"validation NDCG@20","seed":1,"evidence_mode":"pooled_activity_excess","evidence_delta":1.0,"graph_fingerprint":"6b7be789b969cd17a17459deea25bfdf5bfcf518a008b02b7c2925a2d3f88f6c","metrics":{"RECALL@10":0.07624274717468152,"RECALL@20":0.11450039259944982,"NDCG@10":0.04218513022410655,"NDCG@20":0.05208227600724625}},"provenance":"logs/GD6/STAIR-PEEC/stair_icaart/sports_STAIR_ICAART_pooled_activity_excess_d1.0_seed1","fit_seconds":2520.869279},"electronics":{"commit":"5212263e87c5c1cfe6e48c91afa90201a48f5929","python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","gpu":"Tesla T4","v8_config":{"embedding_dim":64,"num_layers":3,"gamma":0.4,"lr":0.001,"weight_decay":0.1,"batch_size":4096,"epochs":500,"beta1":0.9,"beta2":0.999,"mfiles":["textual_modality.pkl","visual_modality.pkl"],"num_neighbors":[5,1],"eval_freq":5,"lambda_nlgcl":0.01,"nlgcl_tau":0.2,"nlgcl_G":1,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"cl_chunk_size":128,"knn_chunk_size":1024,"knn_device":"cpu","cf_block_size":64,"cf_memory_budget_mib":128.0},"icaart_config":{"evidence_mode":"pooled_activity_excess","evidence_activity_bins":"log2","evidence_halves":0,"evidence_seed":null,"evidence_tau":2.0,"evidence_delta":1.0,"evidence_candidate_block":8192,"evidence_placebo_seed":20261016},"features":[{"file":"textual_modality.pkl","sha256":"14a936f18890fbd8b4e6436061b971de878ba79005a2477325d3466e305d635e"},{"file":"visual_modality.pkl","sha256":"13c40105ff6872616e60d0418faa5c4eb4e14e0de1af7d1e71e6820df63dea17"}],"knn":[{"features":"14a936f18890fbd8b4e6436061b971de878ba79005a2477325d3466e305d635e","k":5,"chunk":177,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"2459d9b74920fa59d566d62171cbc1434d1c5e98d263d71c6561c2e7b2c9f695","cache_hit":false,"candidate_sha256":"a7be45316c97702a72dcbce8d1595b2dca971af0a132611646ae430522b09d54"},{"features":"13c40105ff6872616e60d0418faa5c4eb4e14e0de1af7d1e71e6820df63dea17","k":1,"chunk":177,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"6488fd7aaee647ded9774efd93bfe5c61ee399d4c36a23505569352cb8411b75","cache_hit":false,"candidate_sha256":"536c50370c58b370550c0e947d706b0fc3cadbf32d9f243f2ab0fbc5d0f744eb"}],"contract":{"config":{"embedding_dim":64,"num_layers":3,"gamma":0.4,"num_neighbors":[5,1],"mfiles":["textual_modality.pkl","visual_modality.pkl"],"lr":0.001,"weight_decay":0.1,"beta1":0.9,"beta2":0.999,"batch_size":4096,"seed":1,"eval_freq":5,"ranking":"full","which4best":"NDCG@20","num_workers":4,"nlgcl_G":1,"nlgcl_tau":0.2,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"placebo_seed":1,"knn_device":"cpu","knn_chunk_size":1024,"cl_chunk_size":128,"metadata_fingerprint":null},"splits":{"test.txt":"a5f4a1c7eb973d5d6bac5e6f023fcdedf1a66a46805714d233b4eb1fdb468234","train.txt":"495c58aec6c0b23c6780710d58e4718504af26175da8e01855b1f22675062e63","valid.txt":"145ce0ab06bf85260b74ba50829ab5041a2fc5c867d542c81e42da8aefefb51a"},"candidate_hashes":["a7be45316c97702a72dcbce8d1595b2dca971af0a132611646ae430522b09d54","536c50370c58b370550c0e947d706b0fc3cadbf32d9f243f2ab0fbc5d0f744eb"],"sources":{"stair5_v8.py":"dea612a122c82f7cc6c4c1ba069510fa1242fa3819f00b6cbfe0d37b9ed9c502","stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","freerec_compat.py":"de46f3dd9bc11bb69acd5f151d175968e73d980be37ad36d5afc636b0b9c5e90","stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8","stair_icaart.py":"77b862894f9aa91eba8fa96ad54a38eb6e29a25d9e134b7718a58f4bb5851014","stair_icaart_evidence.py":"11487e0fef87077cde7863b58847bcf22adeebef6e5378ae32f71f366d88c41a","stair_icaart_graph.py":"33bd582c8cd3b298eed1ce47ad99837ec9f0cd220778d684ae3d96161a455aad","stair_icaart_config.py":"fe5bcaa711ba64fd40d431db27ae6278e7c937ce05f57786b6f7262668061a51","main_stair_icaart.py":"b95eb5423ac8947da290bd13562e0aeb433adb8742ca0f0246139b4880aeec3f","stair_icaart_reporting.py":"6902360e9ac92dd511ea87e988ad622cae51886eb0ac183b170ce7df1b5065cc"},"data_fingerprint":"84680e889deeecb904f4475620eb0d638da332a95202c193c73c68924b0f0a82","graph_fingerprint":"43b9d0d255d7c7e72322fbeb834b0bc08206fcb236680db373aa30b6b6040242"},"selected_receipt":{"mode":"test","epoch":485,"step":-1,"selected_checkpoint":true,"selection":"validation NDCG@20","seed":1,"evidence_mode":"pooled_activity_excess","evidence_delta":1.0,"graph_fingerprint":"43b9d0d255d7c7e72322fbeb834b0bc08206fcb236680db373aa30b6b6040242","metrics":{"RECALL@10":0.04640053988342894,"RECALL@20":0.06811273359579144,"NDCG@10":0.026213868650622203,"NDCG@20":0.03183061497051809}},"provenance":"logs/GD6/STAIR-PEEC/electronics/stair_icaart/electronics_STAIR_ICAART_pooled_activity_excess_d1.0_seed1","fit_seconds":31999.033542}}')

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False, allow_nan=False) + '\n', encoding='utf-8')
    temporary.replace(path)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


write_json(RUN_ROOT / 'provenance' / 'seed1_reference.json', SEED1_REFERENCE)

WO_EC_GRAPH = {'baby': 'eab753510c9d830425ba0f1f44f6d4964aa0143c574d4707fb2e7c9d93532041', 'sports': 'cc0ff8deb6bbdac1c9295e27781dbb96de422731779794d79b4e1fd355ee75ea', 'electronics': '3de143b8c1d3b5fe5c8107964b1131be71d9b49b2eec36a37d6c160d08bcc22b'}

LEGACY_REUSE_AUDIT = json.loads('{"baby":{"dataset":"baby","arm":"wo_ec","seed":1,"status":"REUSED_TRAINING_PENDING_SELECTED_REEVALUATION","source_directory":"logs/GD5/stair5_v8_artifacts/stair5_v8/baby_WMSG-core_seed1","source_log":"logs/GD5/stair5_v8_artifacts/stair5_v8/baby_WMSG-core_seed1.log","training_commit":"014886e7e7cd03096c724ba67815907e87e05d7c","runtime":{"python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","gpu":"Tesla T4"},"selected_epoch":430,"historical_selected_test_rounded_4dp":{"RECALL@1":0.0128,"RECALL@10":0.0675,"RECALL@20":0.1054,"NDCG@10":0.0365,"NDCG@20":0.0462},"historical_fit_seconds":1158.378381,"checks":{"splits":true,"features":true,"candidate_hashes":true,"data_fingerprint":true,"v8_hyperparameters":true,"common_checkpoint_configuration":true,"native_delta_zero_operator":true,"seed_selection_precision":true,"recorded_sources_verified_at_recorded_commit":true,"all_500_epochs":true,"normal_fit_completion":true},"files":{"manifest.json":{"sha256":"31c04e86e45bfb9afd2c23883608447b4d97c63adf5f2ce88d0ea3ff4d2862c3","bytes":9983},"best_model.pt":{"sha256":"fa19032885f03e28efabcf9e867d15ad48cb1fdfdd2144d7108219e9136e6105","bytes":10914316},"training_telemetry.jsonl":{"sha256":"e338526062e19138a0d97451cb982d36d93a69d44192bf570de5b5cee3d889dd","bytes":178722},"training.log":{"sha256":"aa46fd72f3458f2dccc445d40ce2dd01d7f6d253e50189e06b3fb04a2348ad17","bytes":129058}},"source_hashes":{"models/stair5_v8.py":"c022aeb43c00112df5d93a14724afbc239b5cfee25fc4d9bd8e714301975ae92","models/stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","models/stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","models/stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","models/stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","models/stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","models/stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","models/freerec_compat.py":"708364b1874ab9c1a4d9ce0a21a2e31e57b9bfeac1da4c570af43a28bb80e626","optimizers/stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","optimizers/AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8"},"graph_fingerprint":"eab753510c9d830425ba0f1f44f6d4964aa0143c574d4707fb2e7c9d93532041","legacy_manifest":{"argv":["/kaggle/working/STAIR-Enhanced/main_stair5_v8.py","--config","/kaggle/working/STAIR-Enhanced/configs/Amazon2014Baby_STAIR5_v8.yaml","--root","/kaggle/data","--device","0","--ranking","full","--artifact-dir","/kaggle/working/runs/stair5_v8/20261009_061008_a91d14/baby_WMSG-core_seed1","--v8-arm","WMSG-core","--semantic-mode","weighted","--edge-power","2.0","--edge-floor","0.05","--semantic-mix","1.0","--alignment","off","--anchor-seed","2718","--anchor-per-stratum","1024","--relation-gate","off","--relation-strength","0.0","--relation-threshold","0.7","--relation-temperature","0.1","--bsc-residual","0.0","--lambda-dirichlet","0.0","--seed","1","--eval-freq","5","--eta","0.1","--k-cf","5","--c-min","2","--t-shrinkage","5.0","--cf-block-size","64","--cf-memory-budget-mib","128","--lambda-nlgcl","0.01","--nlgcl-tau","0.2","--nlgcl-G","1","--nlgcl-alpha","0.5","--cl-chunk-size","1024","--knn-chunk-size","1024","--knn-device","cpu","--graph-cache-dir","/kaggle/working/runs/stair5_v8/graph_cache"],"commit":"014886e7e7cd03096c724ba67815907e87e05d7c","python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","device":"cuda:0","cuda_available":true,"gpu":"Tesla T4","arm":"WMSG-core","seed":1,"graph":{"version":8,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"relation_gate":"off","relation_strength":0.0,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"num_users":19445,"num_items":7050,"cache_hit":false,"modality_0_edges_count":35250,"modality_0_cosine_mean":0.733953595161438,"modality_0_cosine_std":0.11293145269155502,"modality_0_nonpositive_cosine_fraction":0.0,"modality_0_weight_mean":0.5738692879676819,"modality_0_weight_min":0.13981308043003082,"modality_0_weight_max":1.0,"modality_1_edges_count":7050,"modality_1_cosine_mean":0.6781584024429321,"modality_1_cosine_std":0.12048749625682831,"modality_1_nonpositive_cosine_fraction":0.0,"modality_1_weight_mean":0.5006951689720154,"modality_1_weight_min":0.17312704026699066,"modality_1_weight_max":1.0,"wmsg_symmetric_nnz":59852,"wmsg_max_weight":2.0,"wmsg_min_weight":0.13981308043003082,"rpg_active":false,"active":true,"effective_eta":0.1,"s8_nnz":84138,"s0_nnz":59852,"cf_nnz":23054,"relative_operator_delta_vs_s0":0.2833131945359231,"graph_build_seconds":0.4915321499993297,"graph_fingerprint":"eab753510c9d830425ba0f1f44f6d4964aa0143c574d4707fb2e7c9d93532041"},"features":[{"file":"textual_modality.pkl","sha256":"ceda3582c04f4e276e8d4bdcc2146f2634f6ca171e1f9dfafeb219dc5cfe96eb"},{"file":"visual_modality.pkl","sha256":"763bf9510c74fd5c6a45ee8164ca9f1ababc5c355c1757fc3acbdcc5d46af0a5"}],"knn":[{"features":"ceda3582c04f4e276e8d4bdcc2146f2634f6ca171e1f9dfafeb219dc5cfe96eb","k":5,"chunk":1024,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"2cb2ad16db4996b65a179bbb221bbf09ac3eb390cf2cacf2cb3b26fd471efa40","cache_hit":false,"candidate_sha256":"5bfeae7c9bb0a96be4ca20803e20d0d84a57068aeb322749bd132f0ce2691827"},{"features":"763bf9510c74fd5c6a45ee8164ca9f1ababc5c355c1757fc3acbdcc5d46af0a5","k":1,"chunk":1024,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"a50d5b1b5a586f38a4a5aec66d128556d43046d899d631a8756b476d47e04586","cache_hit":false,"candidate_sha256":"ce97e02fbb82f94038c40fd2ae8e911c387c951b7710ee29eb2b144fc3cd7b08"}],"sap_audit":{"alignment_mode":"off","status":"BYPASS_V4_PARITY","passed":true,"fallback_reason":null,"num_items":7050,"embedding_dim":64,"fused_formula":"V^{(0)} = (5*Z_t + Z_v) / 6"},"checkpoint_contract":{"version":8,"arm":"WMSG-core","lambda_nlgcl":0.01,"config":{"embedding_dim":64,"num_layers":3,"gamma":0.1,"num_neighbors":[5,1],"mfiles":["textual_modality.pkl","visual_modality.pkl"],"lr":0.001,"weight_decay":0.3,"beta1":0.9,"beta2":0.999,"batch_size":1024,"seed":1,"eval_freq":5,"ranking":"full","which4best":"NDCG@20","num_workers":4,"nlgcl_G":1,"nlgcl_tau":0.2,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"placebo_seed":1,"knn_device":"cpu","knn_chunk_size":1024,"cl_chunk_size":1024,"metadata_fingerprint":null},"data_fingerprint":"de38b811bc05e4838c2f52d025ee3464447a1af4e1a1844747635cddafd13a9e","graph_fingerprint":"eab753510c9d830425ba0f1f44f6d4964aa0143c574d4707fb2e7c9d93532041","sap_audit":{"alignment_mode":"off","status":"BYPASS_V4_PARITY","passed":true,"fallback_reason":null,"num_items":7050,"embedding_dim":64,"fused_formula":"V^{(0)} = (5*Z_t + Z_v) / 6"},"sap_transform_sha256":null,"candidate_hashes":["5bfeae7c9bb0a96be4ca20803e20d0d84a57068aeb322749bd132f0ce2691827","ce97e02fbb82f94038c40fd2ae8e911c387c951b7710ee29eb2b144fc3cd7b08"],"features":[{"file":"textual_modality.pkl","sha256":"ceda3582c04f4e276e8d4bdcc2146f2634f6ca171e1f9dfafeb219dc5cfe96eb"},{"file":"visual_modality.pkl","sha256":"763bf9510c74fd5c6a45ee8164ca9f1ababc5c355c1757fc3acbdcc5d46af0a5"}],"splits":{"test.txt":"307162309a9fae52172dabbbb0f373e845af5e73740d987b70bb0f406d5ac411","train.txt":"bcced50dea427734d76b031f1de140f219d67ba6bb6bc5eac85d7b3be51645bb","valid.txt":"c44687959888d29e2d931aa3de1210f7f708e204d169b7680e5adfcf0a183a3a"},"sources":{"stair5_v8.py":"c022aeb43c00112df5d93a14724afbc239b5cfee25fc4d9bd8e714301975ae92","stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","freerec_compat.py":"708364b1874ab9c1a4d9ce0a21a2e31e57b9bfeac1da4c570af43a28bb80e626","stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8"},"torch":"2.11.0+cu128","freerec":"0.9.7"},"effective_lambda_nlgcl":0.01,"precision":{"tf32":false,"amp":false},"preprocessing_seconds":24.106930313999328,"preprocessing_gpu_memory":{"peak_allocated_bytes":512,"peak_reserved_bytes":2097152},"metadata_fingerprint":null,"ranking":"full","selection":"validation NDCG@20","v8_config":{"embedding_dim":64,"num_layers":3,"gamma":0.1,"lr":0.001,"weight_decay":0.3,"batch_size":1024,"epochs":500,"beta1":0.9,"beta2":0.999,"mfiles":["textual_modality.pkl","visual_modality.pkl"],"num_neighbors":[5,1],"eval_freq":5,"lambda_nlgcl":0.01,"nlgcl_tau":0.2,"nlgcl_G":1,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"cl_chunk_size":1024,"knn_chunk_size":1024,"knn_device":"cpu","cf_block_size":64,"cf_memory_budget_mib":128.0,"v4_support_files":[],"metadata_file":"","placebo_seed":1},"freerec_log_path":"./logs/STAIR5-v8-WMSG-CSE/Amazon2014Baby_550_MMRec/1009065247","test_records":"Native FreeRec final and selected; report selected only."},"full_reference_graph_fingerprint":"b8013795d439064965dcd6946a44751d9a275ca0cefdc5231192066582d757a8","full_reference_baseline_v8_fingerprint":"eab753510c9d830425ba0f1f44f6d4964aa0143c574d4707fb2e7c9d93532041","precision_caveat":"Historical selected TEST line has four decimals. Do not treat summary-at-500 as selected-at-430 for Baby.","evaluation_receipt_status":"NOT_MEASURED_BY_THIS_AUDIT"},"sports":{"dataset":"sports","arm":"wo_ec","seed":1,"status":"REUSED_TRAINING_PENDING_SELECTED_REEVALUATION","source_directory":"logs/GD5/stair5_v8_artifacts/stair5_v8/sports_WMSG-core_seed1","source_log":"logs/GD5/stair5_v8_artifacts/stair5_v8/sports_WMSG-core_seed1.log","training_commit":"014886e7e7cd03096c724ba67815907e87e05d7c","runtime":{"python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","gpu":"Tesla T4"},"selected_epoch":500,"historical_selected_test_rounded_4dp":{"RECALL@1":0.0151,"RECALL@10":0.0762,"RECALL@20":0.1142,"NDCG@10":0.042,"NDCG@20":0.0518},"historical_fit_seconds":2499.692581,"checks":{"splits":true,"features":true,"candidate_hashes":true,"data_fingerprint":true,"v8_hyperparameters":true,"common_checkpoint_configuration":true,"native_delta_zero_operator":true,"seed_selection_precision":true,"recorded_sources_verified_at_recorded_commit":true,"all_500_epochs":true,"normal_fit_completion":true},"files":{"manifest.json":{"sha256":"5d54a92ff79e1b8c00763442105aabd2a0d0eec46eb271bf0d9c5a5a80dc1d26","bytes":10864},"best_model.pt":{"sha256":"f1beede0258de52124b56130f8611f9f32299e571f60789d070808356858d4a1","bytes":22116684},"training_telemetry.jsonl":{"sha256":"08784453e4142815dcf0028d0cb6bd271c7cbd1bf60fffc9355131a6983feaed","bytes":179769},"training.log":{"sha256":"a02f38e5928654f897478e9ae3b9424aeafc638414554677571ca7f0eec74585","bytes":129338}},"source_hashes":{"models/stair5_v8.py":"c022aeb43c00112df5d93a14724afbc239b5cfee25fc4d9bd8e714301975ae92","models/stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","models/stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","models/stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","models/stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","models/stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","models/stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","models/freerec_compat.py":"708364b1874ab9c1a4d9ce0a21a2e31e57b9bfeac1da4c570af43a28bb80e626","optimizers/stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","optimizers/AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8"},"graph_fingerprint":"cc0ff8deb6bbdac1c9295e27781dbb96de422731779794d79b4e1fd355ee75ea","legacy_manifest":{"argv":["/kaggle/working/STAIR-Enhanced/main_stair5_v8.py","--config","/kaggle/working/STAIR-Enhanced/configs/Amazon2014Sports_STAIR5_v8.yaml","--root","/kaggle/data","--device","0","--ranking","full","--artifact-dir","/kaggle/working/runs/stair5_v8/20261009_061008_a91d14/sports_WMSG-core_seed1","--v8-arm","WMSG-core","--semantic-mode","weighted","--edge-power","2.0","--edge-floor","0.05","--semantic-mix","1.0","--alignment","off","--anchor-seed","2718","--anchor-per-stratum","1024","--relation-gate","off","--relation-strength","0.0","--relation-threshold","0.7","--relation-temperature","0.1","--bsc-residual","0.0","--lambda-dirichlet","0.0","--seed","1","--eval-freq","5","--eta","0.1","--k-cf","5","--c-min","2","--t-shrinkage","5.0","--cf-block-size","64","--cf-memory-budget-mib","128","--lambda-nlgcl","0.01","--nlgcl-tau","0.2","--nlgcl-G","1","--nlgcl-alpha","0.5","--cl-chunk-size","1024","--knn-chunk-size","1024","--knn-device","cpu","--graph-cache-dir","/kaggle/working/runs/stair5_v8/graph_cache"],"commit":"014886e7e7cd03096c724ba67815907e87e05d7c","python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","device":"cuda:0","cuda_available":true,"gpu":"Tesla T4","arm":"WMSG-core","seed":1,"graph":{"version":8,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"relation_gate":"off","relation_strength":0.0,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"num_users":35598,"num_items":18357,"cache_hit":false,"modality_0_edges_count":91785,"modality_0_cosine_mean":0.7440291047096252,"modality_0_cosine_std":0.11236167699098587,"modality_0_nonpositive_cosine_fraction":0.0,"modality_0_weight_mean":0.5878943204879761,"modality_0_weight_min":0.1342153549194336,"modality_0_weight_max":1.0,"modality_1_edges_count":18357,"modality_1_cosine_mean":0.7225796580314636,"modality_1_cosine_std":0.13192041218280792,"modality_1_nonpositive_cosine_fraction":0.0,"modality_1_weight_mean":0.562548041343689,"modality_1_weight_min":0.15613286197185516,"modality_1_weight_max":1.0,"wmsg_symmetric_nnz":157744,"wmsg_max_weight":2.0,"wmsg_min_weight":0.1342153549194336,"rpg_active":false,"cf_cache_hit":true,"active":true,"effective_eta":0.1,"s8_nnz":206296,"s0_nnz":157744,"cf_nnz":43334,"relative_operator_delta_vs_s0":0.2968943678218056,"graph_build_seconds":0.5968075470000258,"graph_fingerprint":"cc0ff8deb6bbdac1c9295e27781dbb96de422731779794d79b4e1fd355ee75ea"},"features":[{"file":"textual_modality.pkl","sha256":"ca99972f3a2c7a9ead14e9a0d649351411722bbd98e920b4c5fe9f8ffa1564a2"},{"file":"visual_modality.pkl","sha256":"5d2ea8187bbc52f806616404815930061acefe6cb22f353e3f02f286c95b9959"}],"knn":[{"features":"ca99972f3a2c7a9ead14e9a0d649351411722bbd98e920b4c5fe9f8ffa1564a2","k":5,"chunk":609,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"9dacd55f8d2be69d2bebff457c2c83345135e05f90eb44c1c4f0b8d62b4347e7","cache_hit":true,"loaded_signature":{"features":"ca99972f3a2c7a9ead14e9a0d649351411722bbd98e920b4c5fe9f8ffa1564a2","k":5,"chunk":609,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038"},"candidate_sha256":"bbe9e918209a03b5186a74cbd78ba474f420dfaab4eb6c2095c7c578ac0ca5af","frozen_support":false},{"features":"5d2ea8187bbc52f806616404815930061acefe6cb22f353e3f02f286c95b9959","k":1,"chunk":609,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"9a875c92e35fbe67a4e43ac1298d898fed91f7160d449e4d9e8d5b86899889e5","cache_hit":true,"loaded_signature":{"features":"5d2ea8187bbc52f806616404815930061acefe6cb22f353e3f02f286c95b9959","k":1,"chunk":609,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038"},"candidate_sha256":"e3270692a18ae7d11d50a83d30b7f071f4a095e630714c01b2a0f6b4cff4e8a5","frozen_support":false}],"sap_audit":{"alignment_mode":"off","status":"BYPASS_V4_PARITY","passed":true,"fallback_reason":null,"num_items":18357,"embedding_dim":64,"fused_formula":"V^{(0)} = (5*Z_t + Z_v) / 6"},"checkpoint_contract":{"version":8,"arm":"WMSG-core","lambda_nlgcl":0.01,"config":{"embedding_dim":64,"num_layers":3,"gamma":0.2,"num_neighbors":[5,1],"mfiles":["textual_modality.pkl","visual_modality.pkl"],"lr":0.001,"weight_decay":0.1,"beta1":0.9,"beta2":0.999,"batch_size":1024,"seed":1,"eval_freq":5,"ranking":"full","which4best":"NDCG@20","num_workers":4,"nlgcl_G":1,"nlgcl_tau":0.2,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"placebo_seed":1,"knn_device":"cpu","knn_chunk_size":1024,"cl_chunk_size":1024,"metadata_fingerprint":null},"data_fingerprint":"383929cee0a814ae8211bde6c15ef0abb6ecd4499822bdf5f3579cb87fa04643","graph_fingerprint":"cc0ff8deb6bbdac1c9295e27781dbb96de422731779794d79b4e1fd355ee75ea","sap_audit":{"alignment_mode":"off","status":"BYPASS_V4_PARITY","passed":true,"fallback_reason":null,"num_items":18357,"embedding_dim":64,"fused_formula":"V^{(0)} = (5*Z_t + Z_v) / 6"},"sap_transform_sha256":null,"candidate_hashes":["bbe9e918209a03b5186a74cbd78ba474f420dfaab4eb6c2095c7c578ac0ca5af","e3270692a18ae7d11d50a83d30b7f071f4a095e630714c01b2a0f6b4cff4e8a5"],"features":[{"file":"textual_modality.pkl","sha256":"ca99972f3a2c7a9ead14e9a0d649351411722bbd98e920b4c5fe9f8ffa1564a2"},{"file":"visual_modality.pkl","sha256":"5d2ea8187bbc52f806616404815930061acefe6cb22f353e3f02f286c95b9959"}],"splits":{"test.txt":"218501613d897422d24d2035e268af027f4ca77688a3f836080b197621f5610a","train.txt":"2491704e8543dbdeb9046cec0928c6734aa80521b1026afd53e0d296f4f7427c","valid.txt":"6dabc7efcc14de50909c6b1995f7bf9862d6233a3440d593d389e1bf91fef56e"},"sources":{"stair5_v8.py":"c022aeb43c00112df5d93a14724afbc239b5cfee25fc4d9bd8e714301975ae92","stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","freerec_compat.py":"708364b1874ab9c1a4d9ce0a21a2e31e57b9bfeac1da4c570af43a28bb80e626","stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8"},"torch":"2.11.0+cu128","freerec":"0.9.7"},"effective_lambda_nlgcl":0.01,"precision":{"tf32":false,"amp":false},"preprocessing_seconds":28.933243898,"preprocessing_gpu_memory":{"peak_allocated_bytes":512,"peak_reserved_bytes":2097152},"metadata_fingerprint":null,"ranking":"full","selection":"validation NDCG@20","v8_config":{"embedding_dim":64,"num_layers":3,"gamma":0.2,"lr":0.001,"weight_decay":0.1,"batch_size":1024,"epochs":500,"beta1":0.9,"beta2":0.999,"mfiles":["textual_modality.pkl","visual_modality.pkl"],"num_neighbors":[5,1],"eval_freq":5,"lambda_nlgcl":0.01,"nlgcl_tau":0.2,"nlgcl_G":1,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"cl_chunk_size":1024,"knn_chunk_size":1024,"knn_device":"cpu","cf_block_size":64,"cf_memory_budget_mib":128.0,"v4_support_files":[],"metadata_file":"","placebo_seed":1},"freerec_log_path":"./logs/STAIR5-v8-WMSG-CSE/Amazon2014Sports_550_MMRec/1009061020","test_records":"Native FreeRec final and selected; report selected only."},"full_reference_graph_fingerprint":"6b7be789b969cd17a17459deea25bfdf5bfcf518a008b02b7c2925a2d3f88f6c","full_reference_baseline_v8_fingerprint":"cc0ff8deb6bbdac1c9295e27781dbb96de422731779794d79b4e1fd355ee75ea","precision_caveat":"Historical selected TEST line has four decimals. Do not treat summary-at-500 as selected-at-430 for Baby.","evaluation_receipt_status":"NOT_MEASURED_BY_THIS_AUDIT"},"electronics":{"dataset":"electronics","arm":"wo_ec","seed":1,"status":"REUSED_TRAINING_PENDING_SELECTED_REEVALUATION","source_directory":"logs/GD5/stair5_v8_artifacts/electronics_WMSG/stair5_v8/electronics_WMSG-core_seed1","source_log":"logs/GD5/stair5_v8_artifacts/electronics_WMSG/stair5_v8/electronics_WMSG-core_seed1.log","training_commit":"d401a27511a3f77b13c082d68c7f2359b7e4e61c","runtime":{"python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","gpu":"Tesla T4"},"selected_epoch":500,"historical_selected_test_rounded_4dp":{"RECALL@1":0.0102,"RECALL@10":0.046,"RECALL@20":0.068,"NDCG@10":0.026,"NDCG@20":0.0317},"historical_fit_seconds":31602.465461,"checks":{"splits":true,"features":true,"candidate_hashes":true,"data_fingerprint":true,"v8_hyperparameters":true,"common_checkpoint_configuration":true,"native_delta_zero_operator":true,"seed_selection_precision":true,"recorded_sources_verified_at_recorded_commit":true,"all_500_epochs":true,"normal_fit_completion":true},"files":{"manifest.json":{"sha256":"02aa9fc747d574c4dba723bc27e2b08a076b60df755df7f7689593a5bab5be2d","bytes":10004},"best_model.pt":{"sha256":"b268f27b5c869e6cbe2683a710a9cf9bfc34df59be4c8c685f96b549cb7d706d","bytes":106918220},"training_telemetry.jsonl":{"sha256":"39c4cb5b4b37c79d942ea94402e95271270a8ad3cac1586d20d0e35b376da79c","bytes":179699},"training.log":{"sha256":"3627c726eca21ae25d70a39abeffa87e4017c2ce6d06242ff665e6b5cf13a9e9","bytes":130930}},"source_hashes":{"models/stair5_v8.py":"c022aeb43c00112df5d93a14724afbc239b5cfee25fc4d9bd8e714301975ae92","models/stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","models/stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","models/stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","models/stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","models/stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","models/stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","models/freerec_compat.py":"708364b1874ab9c1a4d9ce0a21a2e31e57b9bfeac1da4c570af43a28bb80e626","optimizers/stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","optimizers/AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8"},"graph_fingerprint":"3de143b8c1d3b5fe5c8107964b1131be71d9b49b2eec36a37d6c160d08bcc22b","legacy_manifest":{"argv":["/kaggle/working/STAIR-Enhanced/main_stair5_v8.py","--config","/kaggle/working/STAIR-Enhanced/configs/Amazon2014Electronics_STAIR5_v8.yaml","--root","/kaggle/data","--device","0","--ranking","full","--artifact-dir","/kaggle/working/runs/stair5_v8/20261009_074535_186774/electronics_WMSG-core_seed1","--v8-arm","WMSG-core","--semantic-mode","weighted","--edge-power","2.0","--edge-floor","0.05","--semantic-mix","1.0","--alignment","off","--anchor-seed","2718","--anchor-per-stratum","1024","--relation-gate","off","--relation-strength","0.0","--relation-threshold","0.7","--relation-temperature","0.1","--bsc-residual","0.0","--lambda-dirichlet","0.0","--seed","1","--eval-freq","5","--eta","0.1","--k-cf","5","--c-min","2","--t-shrinkage","5.0","--cf-block-size","64","--cf-memory-budget-mib","128","--lambda-nlgcl","0.01","--nlgcl-tau","0.2","--nlgcl-G","1","--nlgcl-alpha","0.5","--cl-chunk-size","128","--knn-chunk-size","1024","--knn-device","cpu","--graph-cache-dir","/kaggle/working/runs/stair5_v8/graph_cache"],"commit":"d401a27511a3f77b13c082d68c7f2359b7e4e61c","python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","device":"cuda:0","cuda_available":true,"gpu":"Tesla T4","arm":"WMSG-core","seed":1,"graph":{"version":8,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"relation_gate":"off","relation_strength":0.0,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"num_users":192403,"num_items":63001,"cache_hit":false,"modality_0_edges_count":315005,"modality_0_cosine_mean":0.7866738438606262,"modality_0_cosine_std":0.1035960391163826,"modality_0_nonpositive_cosine_fraction":0.0,"modality_0_weight_mean":0.6481084227561951,"modality_0_weight_min":0.11206944286823273,"modality_0_weight_max":1.0,"modality_1_edges_count":63001,"modality_1_cosine_mean":0.7869994640350342,"modality_1_cosine_std":0.12887389957904816,"modality_1_nonpositive_cosine_fraction":0.0,"modality_1_weight_mean":0.6541777849197388,"modality_1_weight_min":0.16854384541511536,"modality_1_weight_max":1.0,"wmsg_symmetric_nnz":542712,"wmsg_max_weight":2.0,"wmsg_min_weight":0.11206944286823273,"rpg_active":false,"active":true,"effective_eta":0.1,"s8_nnz":739381,"s0_nnz":542712,"cf_nnz":174488,"relative_operator_delta_vs_s0":0.28176892036927004,"graph_build_seconds":4.608284074999915,"graph_fingerprint":"3de143b8c1d3b5fe5c8107964b1131be71d9b49b2eec36a37d6c160d08bcc22b"},"features":[{"file":"textual_modality.pkl","sha256":"14a936f18890fbd8b4e6436061b971de878ba79005a2477325d3466e305d635e"},{"file":"visual_modality.pkl","sha256":"13c40105ff6872616e60d0418faa5c4eb4e14e0de1af7d1e71e6820df63dea17"}],"knn":[{"features":"14a936f18890fbd8b4e6436061b971de878ba79005a2477325d3466e305d635e","k":5,"chunk":177,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"2459d9b74920fa59d566d62171cbc1434d1c5e98d263d71c6561c2e7b2c9f695","cache_hit":false,"candidate_sha256":"a7be45316c97702a72dcbce8d1595b2dca971af0a132611646ae430522b09d54"},{"features":"13c40105ff6872616e60d0418faa5c4eb4e14e0de1af7d1e71e6820df63dea17","k":1,"chunk":177,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"6488fd7aaee647ded9774efd93bfe5c61ee399d4c36a23505569352cb8411b75","cache_hit":false,"candidate_sha256":"536c50370c58b370550c0e947d706b0fc3cadbf32d9f243f2ab0fbc5d0f744eb"}],"sap_audit":{"alignment_mode":"off","status":"BYPASS_V4_PARITY","passed":true,"fallback_reason":null,"num_items":63001,"embedding_dim":64,"fused_formula":"V^{(0)} = (5*Z_t + Z_v) / 6"},"checkpoint_contract":{"version":8,"arm":"WMSG-core","lambda_nlgcl":0.01,"config":{"embedding_dim":64,"num_layers":3,"gamma":0.4,"num_neighbors":[5,1],"mfiles":["textual_modality.pkl","visual_modality.pkl"],"lr":0.001,"weight_decay":0.1,"beta1":0.9,"beta2":0.999,"batch_size":4096,"seed":1,"eval_freq":5,"ranking":"full","which4best":"NDCG@20","num_workers":4,"nlgcl_G":1,"nlgcl_tau":0.2,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"placebo_seed":1,"knn_device":"cpu","knn_chunk_size":1024,"cl_chunk_size":128,"metadata_fingerprint":null},"data_fingerprint":"84680e889deeecb904f4475620eb0d638da332a95202c193c73c68924b0f0a82","graph_fingerprint":"3de143b8c1d3b5fe5c8107964b1131be71d9b49b2eec36a37d6c160d08bcc22b","sap_audit":{"alignment_mode":"off","status":"BYPASS_V4_PARITY","passed":true,"fallback_reason":null,"num_items":63001,"embedding_dim":64,"fused_formula":"V^{(0)} = (5*Z_t + Z_v) / 6"},"sap_transform_sha256":null,"candidate_hashes":["a7be45316c97702a72dcbce8d1595b2dca971af0a132611646ae430522b09d54","536c50370c58b370550c0e947d706b0fc3cadbf32d9f243f2ab0fbc5d0f744eb"],"features":[{"file":"textual_modality.pkl","sha256":"14a936f18890fbd8b4e6436061b971de878ba79005a2477325d3466e305d635e"},{"file":"visual_modality.pkl","sha256":"13c40105ff6872616e60d0418faa5c4eb4e14e0de1af7d1e71e6820df63dea17"}],"splits":{"test.txt":"a5f4a1c7eb973d5d6bac5e6f023fcdedf1a66a46805714d233b4eb1fdb468234","train.txt":"495c58aec6c0b23c6780710d58e4718504af26175da8e01855b1f22675062e63","valid.txt":"145ce0ab06bf85260b74ba50829ab5041a2fc5c867d542c81e42da8aefefb51a"},"sources":{"stair5_v8.py":"c022aeb43c00112df5d93a14724afbc239b5cfee25fc4d9bd8e714301975ae92","stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","freerec_compat.py":"708364b1874ab9c1a4d9ce0a21a2e31e57b9bfeac1da4c570af43a28bb80e626","stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8"},"torch":"2.11.0+cu128","freerec":"0.9.7"},"effective_lambda_nlgcl":0.01,"precision":{"tf32":false,"amp":false},"preprocessing_seconds":280.474905628,"preprocessing_gpu_memory":{"peak_allocated_bytes":512,"peak_reserved_bytes":2097152},"metadata_fingerprint":null,"ranking":"full","selection":"validation NDCG@20","v8_config":{"embedding_dim":64,"num_layers":3,"gamma":0.4,"lr":0.001,"weight_decay":0.1,"batch_size":4096,"epochs":500,"beta1":0.9,"beta2":0.999,"mfiles":["textual_modality.pkl","visual_modality.pkl"],"num_neighbors":[5,1],"eval_freq":5,"lambda_nlgcl":0.01,"nlgcl_tau":0.2,"nlgcl_G":1,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"cl_chunk_size":128,"knn_chunk_size":1024,"knn_device":"cpu","cf_block_size":64,"cf_memory_budget_mib":128.0,"v4_support_files":[],"metadata_file":"","placebo_seed":1},"freerec_log_path":"./logs/STAIR5-v8-WMSG-CSE/Amazon2014Electronics_550_MMRec/1009074547","test_records":"Native FreeRec final and selected; report selected only."},"full_reference_graph_fingerprint":"43b9d0d255d7c7e72322fbeb834b0bc08206fcb236680db373aa30b6b6040242","full_reference_baseline_v8_fingerprint":"3de143b8c1d3b5fe5c8107964b1131be71d9b49b2eec36a37d6c160d08bcc22b","precision_caveat":"Historical selected TEST line has four decimals. Do not treat summary-at-500 as selected-at-430 for Baby.","evaluation_receipt_status":"NOT_MEASURED_BY_THIS_AUDIT"}}')

VERIFIED_WO_EC_SEED1 = json.loads('{"baby":{"receipt":{"method":"ECLIPSE","arm":"wo_ec","dataset":"baby","seed":1,"mode":"test","epoch":430,"selected_checkpoint":true,"selection":"validation NDCG@20","metrics":{"RECALL@1":0.012814899254853697,"RECALL@10":0.06748052135121396,"RECALL@20":0.10543584425733217,"NDCG@10":0.03647646875829309,"NDCG@20":0.04624348995284177},"evidence_mode":"pooled_activity_excess","evidence_delta":0.0,"graph_fingerprint":"eab753510c9d830425ba0f1f44f6d4964aa0143c574d4707fb2e7c9d93532041","training_reused":true,"checkpoint_sha256":"fa19032885f03e28efabcf9e867d15ad48cb1fdfdd2144d7108219e9136e6105","checkpoint_path":"/kaggle/working/eclipse_runs/eclipse_wo_ec_seed1_reuse_20261018T152158Z_69603726/artifacts/baby_eclipse_wo_ec_seed1/best_model.pt","source_hashes":{"models/stair5_v8.py":"c022aeb43c00112df5d93a14724afbc239b5cfee25fc4d9bd8e714301975ae92","models/stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","models/stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","models/stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","models/stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","models/stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","models/stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","models/freerec_compat.py":"708364b1874ab9c1a4d9ce0a21a2e31e57b9bfeac1da4c570af43a28bb80e626","optimizers/stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","optimizers/AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8"},"training_commit":"014886e7e7cd03096c724ba67815907e87e05d7c","evaluation_runtime":{"python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7"},"runtime_deviations":{},"provenance":"Original native uncalibrated checkpoint, operator-equivalent to exact delta=0 bypass. No native metadata rewritten; TEST evaluator/scorer/seen masking unchanged.","metrics_precision":"native AverageMeter averages before step reset"},"provenance":"logs/ICAART/eclipse_wo_ec_seed1_reuse_20261018T152158Z_69603726/eclipse_wo_ec_seed1_reuse_20261018T152158Z_69603726/artifacts/baby_eclipse_wo_ec_seed1/selected_metrics.json","receipt_sha256":"1aee747c4f14834966ff92e809088b4a61f2537f55128acbda7b11c663cc51fb","manifest_sha256":"31c04e86e45bfb9afd2c23883608447b4d97c63adf5f2ce88d0ea3ff4d2862c3","status":"REUSED_VERIFIED","verification":"File/receipt/source/runtime audit of user-executed native evaluation; no local model rerun","full_selected_epoch":480,"full_receipt_sha256":"867447ba932519ac03a790c746b2bcc6e5e84327835ae62208b5a7e8a2bd5f55"},"sports":{"receipt":{"method":"ECLIPSE","arm":"wo_ec","dataset":"sports","seed":1,"mode":"test","epoch":500,"selected_checkpoint":true,"selection":"validation NDCG@20","metrics":{"RECALL@1":0.01509081346127108,"RECALL@10":0.07618511536361273,"RECALL@20":0.11424803785061768,"NDCG@10":0.04196987145603008,"NDCG@20":0.051815715919244734},"evidence_mode":"pooled_activity_excess","evidence_delta":0.0,"graph_fingerprint":"cc0ff8deb6bbdac1c9295e27781dbb96de422731779794d79b4e1fd355ee75ea","training_reused":true,"checkpoint_sha256":"f1beede0258de52124b56130f8611f9f32299e571f60789d070808356858d4a1","checkpoint_path":"/kaggle/working/eclipse_runs/eclipse_wo_ec_seed1_reuse_20261018T152158Z_69603726/artifacts/sports_eclipse_wo_ec_seed1/best_model.pt","source_hashes":{"models/stair5_v8.py":"c022aeb43c00112df5d93a14724afbc239b5cfee25fc4d9bd8e714301975ae92","models/stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","models/stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","models/stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","models/stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","models/stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","models/stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","models/freerec_compat.py":"708364b1874ab9c1a4d9ce0a21a2e31e57b9bfeac1da4c570af43a28bb80e626","optimizers/stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","optimizers/AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8"},"training_commit":"014886e7e7cd03096c724ba67815907e87e05d7c","evaluation_runtime":{"python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7"},"runtime_deviations":{},"provenance":"Original native uncalibrated checkpoint, operator-equivalent to exact delta=0 bypass. No native metadata rewritten; TEST evaluator/scorer/seen masking unchanged.","metrics_precision":"native AverageMeter averages before step reset"},"provenance":"logs/ICAART/eclipse_wo_ec_seed1_reuse_20261018T152158Z_69603726/eclipse_wo_ec_seed1_reuse_20261018T152158Z_69603726/artifacts/sports_eclipse_wo_ec_seed1/selected_metrics.json","receipt_sha256":"e32b139e99f35e43725678675c764d004f47541d952364b39db3cca468599d76","manifest_sha256":"5d54a92ff79e1b8c00763442105aabd2a0d0eec46eb271bf0d9c5a5a80dc1d26","status":"REUSED_VERIFIED","verification":"File/receipt/source/runtime audit of user-executed native evaluation; no local model rerun","full_selected_epoch":500,"full_receipt_sha256":"eb57e848f319b0019433f80803f86df34e652b78e3b871183202790628059aa7"},"electronics":{"receipt":{"method":"ECLIPSE","arm":"wo_ec","dataset":"electronics","seed":1,"mode":"test","epoch":500,"selected_checkpoint":true,"selection":"validation NDCG@20","metrics":{"RECALL@1":0.010154512455393701,"RECALL@10":0.04603367736138044,"RECALL@20":0.0679609793540499,"NDCG@10":0.026003775841738932,"NDCG@20":0.031680962471178686},"evidence_mode":"pooled_activity_excess","evidence_delta":0.0,"graph_fingerprint":"3de143b8c1d3b5fe5c8107964b1131be71d9b49b2eec36a37d6c160d08bcc22b","training_reused":true,"checkpoint_sha256":"b268f27b5c869e6cbe2683a710a9cf9bfc34df59be4c8c685f96b549cb7d706d","checkpoint_path":"/kaggle/working/eclipse_runs/eclipse_wo_ec_seed1_reuse_20261018T152158Z_69603726/artifacts/electronics_eclipse_wo_ec_seed1/best_model.pt","source_hashes":{"models/stair5_v8.py":"c022aeb43c00112df5d93a14724afbc239b5cfee25fc4d9bd8e714301975ae92","models/stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","models/stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","models/stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","models/stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","models/stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","models/stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","models/freerec_compat.py":"708364b1874ab9c1a4d9ce0a21a2e31e57b9bfeac1da4c570af43a28bb80e626","optimizers/stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","optimizers/AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8"},"training_commit":"d401a27511a3f77b13c082d68c7f2359b7e4e61c","evaluation_runtime":{"python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7"},"runtime_deviations":{},"provenance":"Original native uncalibrated checkpoint, operator-equivalent to exact delta=0 bypass. No native metadata rewritten; TEST evaluator/scorer/seen masking unchanged.","metrics_precision":"native AverageMeter averages before step reset"},"provenance":"logs/ICAART/eclipse_wo_ec_seed1_reuse_20261018T152158Z_69603726/eclipse_wo_ec_seed1_reuse_20261018T152158Z_69603726/artifacts/electronics_eclipse_wo_ec_seed1/selected_metrics.json","receipt_sha256":"4546683f248b9616f99018f766e59ca5b1aaed13ff9dc9ed57c568390283e329","manifest_sha256":"02aa9fc747d574c4dba723bc27e2b08a076b60df755df7f7689593a5bab5be2d","status":"REUSED_VERIFIED","verification":"File/receipt/source/runtime audit of user-executed native evaluation; no local model rerun","full_selected_epoch":485,"full_receipt_sha256":"f8db7aa2200686c954496d5e5577eaf2bdfe5781b6e25c13a0b1440487ffe971"}}')


# Measured w/o EC seed-2 results: aggregation only, never training inputs.
VERIFIED_WO_EC_SEED2 = json.loads('{"baby":{"status":"VERIFIED_ARTIFACT","receipt":{"mode":"test","epoch":420,"step":-1,"selected_checkpoint":true,"selection":"validation NDCG@20","seed":2,"evidence_mode":"pooled_activity_excess","evidence_delta":0.0,"graph_fingerprint":"eab753510c9d830425ba0f1f44f6d4964aa0143c574d4707fb2e7c9d93532041","metrics":{"RECALL@10":0.06732501574962235,"RECALL@20":0.10360957254983981,"NDCG@10":0.03662976031075501,"NDCG@20":0.04592250146012007}},"manifest_projection":{"seed":2,"selection":"validation NDCG@20","commit":"5212263e87c5c1cfe6e48c91afa90201a48f5929","python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","gpu":"Tesla T4","v8_config":{"embedding_dim":64,"num_layers":3,"gamma":0.1,"lr":0.001,"weight_decay":0.3,"batch_size":1024,"epochs":500,"beta1":0.9,"beta2":0.999,"mfiles":["textual_modality.pkl","visual_modality.pkl"],"num_neighbors":[5,1],"eval_freq":5,"lambda_nlgcl":0.01,"nlgcl_tau":0.2,"nlgcl_G":1,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"cl_chunk_size":1024,"knn_chunk_size":1024,"knn_device":"cpu","cf_block_size":64,"cf_memory_budget_mib":128.0},"icaart_config":{"evidence_mode":"pooled_activity_excess","evidence_activity_bins":"log2","evidence_halves":0,"evidence_seed":null,"evidence_tau":2.0,"evidence_delta":0.0,"evidence_candidate_block":8192,"evidence_placebo_seed":20261016},"features":[{"file":"textual_modality.pkl","sha256":"ceda3582c04f4e276e8d4bdcc2146f2634f6ca171e1f9dfafeb219dc5cfe96eb"},{"file":"visual_modality.pkl","sha256":"763bf9510c74fd5c6a45ee8164ca9f1ababc5c355c1757fc3acbdcc5d46af0a5"}],"precision":{"tf32":false,"amp":false},"checkpoint_contract":{"config":{"embedding_dim":64,"num_layers":3,"gamma":0.1,"num_neighbors":[5,1],"mfiles":["textual_modality.pkl","visual_modality.pkl"],"lr":0.001,"weight_decay":0.3,"beta1":0.9,"beta2":0.999,"batch_size":1024,"seed":2,"eval_freq":5,"ranking":"full","which4best":"NDCG@20","num_workers":4,"nlgcl_G":1,"nlgcl_tau":0.2,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"placebo_seed":1,"knn_device":"cpu","knn_chunk_size":1024,"cl_chunk_size":1024,"metadata_fingerprint":null},"splits":{"test.txt":"307162309a9fae52172dabbbb0f373e845af5e73740d987b70bb0f406d5ac411","train.txt":"bcced50dea427734d76b031f1de140f219d67ba6bb6bc5eac85d7b3be51645bb","valid.txt":"c44687959888d29e2d931aa3de1210f7f708e204d169b7680e5adfcf0a183a3a"},"candidate_hashes":["5bfeae7c9bb0a96be4ca20803e20d0d84a57068aeb322749bd132f0ce2691827","ce97e02fbb82f94038c40fd2ae8e911c387c951b7710ee29eb2b144fc3cd7b08"],"data_fingerprint":"de38b811bc05e4838c2f52d025ee3464447a1af4e1a1844747635cddafd13a9e","graph_fingerprint":"eab753510c9d830425ba0f1f44f6d4964aa0143c574d4707fb2e7c9d93532041","sources":{"stair5_v8.py":"dea612a122c82f7cc6c4c1ba069510fa1242fa3819f00b6cbfe0d37b9ed9c502","stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","freerec_compat.py":"de46f3dd9bc11bb69acd5f151d175968e73d980be37ad36d5afc636b0b9c5e90","stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8","stair_icaart.py":"77b862894f9aa91eba8fa96ad54a38eb6e29a25d9e134b7718a58f4bb5851014","stair_icaart_evidence.py":"11487e0fef87077cde7863b58847bcf22adeebef6e5378ae32f71f366d88c41a","stair_icaart_graph.py":"33bd582c8cd3b298eed1ce47ad99837ec9f0cd220778d684ae3d96161a455aad","stair_icaart_config.py":"fe5bcaa711ba64fd40d431db27ae6278e7c937ce05f57786b6f7262668061a51","main_stair_icaart.py":"b95eb5423ac8947da290bd13562e0aeb433adb8742ca0f0246139b4880aeec3f","stair_icaart_reporting.py":"6902360e9ac92dd511ea87e988ad622cae51886eb0ac183b170ce7df1b5065cc"}},"graph":{"icaart":{"exact_v8_bypass":true,"inactive_reason":"delta_zero","relative_operator_dose":0.0,"relative_polynomial_dose":0.0}}},"receipt_sha256":"519d5b9ac0d3df3690bf41a40bfb6a37ea8fa695b4175860855e29dc81960e8b","manifest_sha256":"f50eadb2b9f776a6764ed897af61edd0329aa7e3af92d79131f4bf6a9cf3b8c6","receipt_canonical_sha256":"246569e5447aa121ef8a515c77d2d37247f9005997f5dce327379ebc265820f3","manifest_projection_canonical_sha256":"9a5e18c18b2fe1b9b0609df6402d252fb869a03502e28476bd99bfcc994b30d6","checkpoint_sha256":"3819ed92e6978b6e3d9b51db09fe81f0fa06bcc0e2ec0ea95e10cfe54c46ef60","provenance":"logs/ICAART/eclipse_wo_ec_seed2/artifacts/baby_eclipse_wo_ec_seed2/selected_metrics.json","audit_provenance":"reports/eclipse_wo_ec_seed2_audit/acceptance.json","scope":"Audited supplied receipts and checkpoint bytes; no local checkpoint load or model rerun."},"sports":{"status":"VERIFIED_ARTIFACT","receipt":{"mode":"test","epoch":470,"step":-1,"selected_checkpoint":true,"selection":"validation NDCG@20","seed":2,"evidence_mode":"pooled_activity_excess","evidence_delta":0.0,"graph_fingerprint":"cc0ff8deb6bbdac1c9295e27781dbb96de422731779794d79b4e1fd355ee75ea","metrics":{"RECALL@10":0.07614413708661057,"RECALL@20":0.1143237173379797,"NDCG@10":0.041547027886453,"NDCG@20":0.05143965237900423}},"manifest_projection":{"seed":2,"selection":"validation NDCG@20","commit":"5212263e87c5c1cfe6e48c91afa90201a48f5929","python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","gpu":"Tesla T4","v8_config":{"embedding_dim":64,"num_layers":3,"gamma":0.2,"lr":0.001,"weight_decay":0.1,"batch_size":1024,"epochs":500,"beta1":0.9,"beta2":0.999,"mfiles":["textual_modality.pkl","visual_modality.pkl"],"num_neighbors":[5,1],"eval_freq":5,"lambda_nlgcl":0.01,"nlgcl_tau":0.2,"nlgcl_G":1,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"cl_chunk_size":1024,"knn_chunk_size":1024,"knn_device":"cpu","cf_block_size":64,"cf_memory_budget_mib":128.0},"icaart_config":{"evidence_mode":"pooled_activity_excess","evidence_activity_bins":"log2","evidence_halves":0,"evidence_seed":null,"evidence_tau":2.0,"evidence_delta":0.0,"evidence_candidate_block":8192,"evidence_placebo_seed":20261016},"features":[{"file":"textual_modality.pkl","sha256":"ca99972f3a2c7a9ead14e9a0d649351411722bbd98e920b4c5fe9f8ffa1564a2"},{"file":"visual_modality.pkl","sha256":"5d2ea8187bbc52f806616404815930061acefe6cb22f353e3f02f286c95b9959"}],"precision":{"tf32":false,"amp":false},"checkpoint_contract":{"config":{"embedding_dim":64,"num_layers":3,"gamma":0.2,"num_neighbors":[5,1],"mfiles":["textual_modality.pkl","visual_modality.pkl"],"lr":0.001,"weight_decay":0.1,"beta1":0.9,"beta2":0.999,"batch_size":1024,"seed":2,"eval_freq":5,"ranking":"full","which4best":"NDCG@20","num_workers":4,"nlgcl_G":1,"nlgcl_tau":0.2,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"placebo_seed":1,"knn_device":"cpu","knn_chunk_size":1024,"cl_chunk_size":1024,"metadata_fingerprint":null},"splits":{"test.txt":"218501613d897422d24d2035e268af027f4ca77688a3f836080b197621f5610a","train.txt":"2491704e8543dbdeb9046cec0928c6734aa80521b1026afd53e0d296f4f7427c","valid.txt":"6dabc7efcc14de50909c6b1995f7bf9862d6233a3440d593d389e1bf91fef56e"},"candidate_hashes":["bbe9e918209a03b5186a74cbd78ba474f420dfaab4eb6c2095c7c578ac0ca5af","e3270692a18ae7d11d50a83d30b7f071f4a095e630714c01b2a0f6b4cff4e8a5"],"data_fingerprint":"383929cee0a814ae8211bde6c15ef0abb6ecd4499822bdf5f3579cb87fa04643","graph_fingerprint":"cc0ff8deb6bbdac1c9295e27781dbb96de422731779794d79b4e1fd355ee75ea","sources":{"stair5_v8.py":"dea612a122c82f7cc6c4c1ba069510fa1242fa3819f00b6cbfe0d37b9ed9c502","stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","freerec_compat.py":"de46f3dd9bc11bb69acd5f151d175968e73d980be37ad36d5afc636b0b9c5e90","stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8","stair_icaart.py":"77b862894f9aa91eba8fa96ad54a38eb6e29a25d9e134b7718a58f4bb5851014","stair_icaart_evidence.py":"11487e0fef87077cde7863b58847bcf22adeebef6e5378ae32f71f366d88c41a","stair_icaart_graph.py":"33bd582c8cd3b298eed1ce47ad99837ec9f0cd220778d684ae3d96161a455aad","stair_icaart_config.py":"fe5bcaa711ba64fd40d431db27ae6278e7c937ce05f57786b6f7262668061a51","main_stair_icaart.py":"b95eb5423ac8947da290bd13562e0aeb433adb8742ca0f0246139b4880aeec3f","stair_icaart_reporting.py":"6902360e9ac92dd511ea87e988ad622cae51886eb0ac183b170ce7df1b5065cc"}},"graph":{"icaart":{"exact_v8_bypass":true,"inactive_reason":"delta_zero","relative_operator_dose":0.0,"relative_polynomial_dose":0.0}}},"receipt_sha256":"e6c640bba31c593284b764af8a5ef22502df7e1e9dc9ff6d2859ab3ef27c237d","manifest_sha256":"7a7293ffe9e49af8960f6a4b46e5525c9593f1e19fc83c3eb15596c677407685","receipt_canonical_sha256":"6ad647b3bb7174edda93e54f6ccfb81c6fb81235638c961ae039b9e002d3250b","manifest_projection_canonical_sha256":"eeee8c07f772352d67650d820ac976f938052378e096139f096dc05961ea4dd7","checkpoint_sha256":"58c446d3d4ad2a472f0f37bda56348b4fcb252897d428e281bdf96e3ff70f351","provenance":"logs/ICAART/eclipse_wo_ec_seed2/artifacts/sports_eclipse_wo_ec_seed2/selected_metrics.json","audit_provenance":"reports/eclipse_wo_ec_seed2_audit/acceptance.json","scope":"Audited supplied receipts and checkpoint bytes; no local checkpoint load or model rerun."},"electronics":{"status":"VERIFIED_ARTIFACT","receipt":{"mode":"test","epoch":480,"step":-1,"selected_checkpoint":true,"selection":"validation NDCG@20","seed":2,"evidence_mode":"pooled_activity_excess","evidence_delta":0.0,"graph_fingerprint":"3de143b8c1d3b5fe5c8107964b1131be71d9b49b2eec36a37d6c160d08bcc22b","metrics":{"RECALL@10":0.04604752569419353,"RECALL@20":0.06816746284732295,"NDCG@10":0.02601073558484886,"NDCG@20":0.031735183034429335}},"manifest_projection":{"seed":2,"selection":"validation NDCG@20","commit":"5212263e87c5c1cfe6e48c91afa90201a48f5929","python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","gpu":"Tesla T4","v8_config":{"embedding_dim":64,"num_layers":3,"gamma":0.4,"lr":0.001,"weight_decay":0.1,"batch_size":4096,"epochs":500,"beta1":0.9,"beta2":0.999,"mfiles":["textual_modality.pkl","visual_modality.pkl"],"num_neighbors":[5,1],"eval_freq":5,"lambda_nlgcl":0.01,"nlgcl_tau":0.2,"nlgcl_G":1,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"cl_chunk_size":128,"knn_chunk_size":1024,"knn_device":"cpu","cf_block_size":64,"cf_memory_budget_mib":128.0},"icaart_config":{"evidence_mode":"pooled_activity_excess","evidence_activity_bins":"log2","evidence_halves":0,"evidence_seed":null,"evidence_tau":2.0,"evidence_delta":0.0,"evidence_candidate_block":8192,"evidence_placebo_seed":20261016},"features":[{"file":"textual_modality.pkl","sha256":"14a936f18890fbd8b4e6436061b971de878ba79005a2477325d3466e305d635e"},{"file":"visual_modality.pkl","sha256":"13c40105ff6872616e60d0418faa5c4eb4e14e0de1af7d1e71e6820df63dea17"}],"precision":{"tf32":false,"amp":false},"checkpoint_contract":{"config":{"embedding_dim":64,"num_layers":3,"gamma":0.4,"num_neighbors":[5,1],"mfiles":["textual_modality.pkl","visual_modality.pkl"],"lr":0.001,"weight_decay":0.1,"beta1":0.9,"beta2":0.999,"batch_size":4096,"seed":2,"eval_freq":5,"ranking":"full","which4best":"NDCG@20","num_workers":4,"nlgcl_G":1,"nlgcl_tau":0.2,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"placebo_seed":1,"knn_device":"cpu","knn_chunk_size":1024,"cl_chunk_size":128,"metadata_fingerprint":null},"splits":{"test.txt":"a5f4a1c7eb973d5d6bac5e6f023fcdedf1a66a46805714d233b4eb1fdb468234","train.txt":"495c58aec6c0b23c6780710d58e4718504af26175da8e01855b1f22675062e63","valid.txt":"145ce0ab06bf85260b74ba50829ab5041a2fc5c867d542c81e42da8aefefb51a"},"candidate_hashes":["a7be45316c97702a72dcbce8d1595b2dca971af0a132611646ae430522b09d54","536c50370c58b370550c0e947d706b0fc3cadbf32d9f243f2ab0fbc5d0f744eb"],"data_fingerprint":"84680e889deeecb904f4475620eb0d638da332a95202c193c73c68924b0f0a82","graph_fingerprint":"3de143b8c1d3b5fe5c8107964b1131be71d9b49b2eec36a37d6c160d08bcc22b","sources":{"stair5_v8.py":"dea612a122c82f7cc6c4c1ba069510fa1242fa3819f00b6cbfe0d37b9ed9c502","stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","freerec_compat.py":"de46f3dd9bc11bb69acd5f151d175968e73d980be37ad36d5afc636b0b9c5e90","stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8","stair_icaart.py":"77b862894f9aa91eba8fa96ad54a38eb6e29a25d9e134b7718a58f4bb5851014","stair_icaart_evidence.py":"11487e0fef87077cde7863b58847bcf22adeebef6e5378ae32f71f366d88c41a","stair_icaart_graph.py":"33bd582c8cd3b298eed1ce47ad99837ec9f0cd220778d684ae3d96161a455aad","stair_icaart_config.py":"fe5bcaa711ba64fd40d431db27ae6278e7c937ce05f57786b6f7262668061a51","main_stair_icaart.py":"b95eb5423ac8947da290bd13562e0aeb433adb8742ca0f0246139b4880aeec3f","stair_icaart_reporting.py":"6902360e9ac92dd511ea87e988ad622cae51886eb0ac183b170ce7df1b5065cc"}},"graph":{"icaart":{"exact_v8_bypass":true,"inactive_reason":"delta_zero","relative_operator_dose":0.0,"relative_polynomial_dose":0.0}}},"receipt_sha256":"a6d845dced5e7aa985e740326ae8ecc3208ed4ef84339851f80f605749534683","manifest_sha256":"694b76e7f853a2ec6dcfd6915ee5b6740f9d6aabf8f48f8809f0fe44f5441a86","receipt_canonical_sha256":"156107233e234d013ab139cc77c0e59429cd7b9688c602c23772fa752dc1fe26","manifest_projection_canonical_sha256":"1b4ae33637fc05c57141eec961b1cb8f17adcf21e8cf23477f6bc7391a743eef","checkpoint_sha256":"6c62eaaac3c57493a21039cc34ae7c1c05e63101f494e7155f2c98b8001e51ac","provenance":"logs/ICAART/eclipse_wo_ec_seed2/artifacts/electronics_eclipse_wo_ec_seed2/selected_metrics.json","audit_provenance":"reports/eclipse_wo_ec_seed2_audit/acceptance.json","scope":"Audited supplied receipts and checkpoint bytes; no local checkpoint load or model rerun."}}')
write_json(RUN_ROOT / "provenance" / "wo_ec_seed2_reference.json", VERIFIED_WO_EC_SEED2)


## 3. Môi trường và mã nguồn

Giữ CUDA Torch do Kaggle cung cấp, không tự nâng/hạ Torch. Checkout commit seed 1 chỉ trong clone mới; không reset checkout có sẵn. Các file thuật toán phải khớp source digests seed 1. Khác phiên bản Torch/FreeRec có thể đổi candidate ordering và hành vi evaluator: mặc định dừng để tránh gọi một thay đổi môi trường là thí nghiệm chỉ đổi seed.


In [ ]:
import importlib.metadata
import importlib.util
import subprocess

if REPO_OVERRIDE is not None:
    REPO_ROOT = Path(REPO_OVERRIDE).expanduser().resolve()
else:
    candidates = [Path.cwd(), WORK_DIR / 'STAIR-Enhanced', Path.home() / 'STAIR-Enhanced']
    REPO_ROOT = next((p.resolve() for p in candidates if (p / 'main_stair_icaart.py').is_file()), None)
    if REPO_ROOT is None:
        REPO_ROOT = WORK_DIR / 'STAIR-Enhanced'
        if REPO_ROOT.exists():
            raise FileExistsError(f'Existing directory is not a compatible repository: {REPO_ROOT}')
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', '--depth', '1', 'origin', RUNTIME_COMMIT], check=True)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '--detach', RUNTIME_COMMIT], check=True)
if not (REPO_ROOT / 'main_stair_icaart.py').is_file():
    raise FileNotFoundError(REPO_ROOT / 'main_stair_icaart.py')

ENV = os.environ.copy()
ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [str(REPO_ROOT), ENV.get('PYTHONPATH', '')]))
ENV['PYTHONUTF8'] = '1'
ENV['PYTHONIOENCODING'] = 'utf-8'
ENV['PYTHONHASHSEED'] = str(SEED)
ENV['CUDA_VISIBLE_DEVICES'] = str(GPU_INDEX)
ENV.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

# Do not import Torch into the notebook kernel: all CUDA work belongs to the training subprocess.
if INSTALL_MISSING_DEPENDENCIES:
    packages = {
        'freerec': ('freerec==0.9.7', True),
        'torch_geometric': ('torch-geometric', True),
        'numpy': ('numpy', False), 'scipy': ('scipy', False),
        'yaml': ('PyYAML', False), 'pandas': ('pandas', False),
        'matplotlib': ('matplotlib', False), 'prettytable': ('prettytable', False),
        'sklearn': ('scikit-learn', False), 'psutil': ('psutil', False),
        'kagglehub': ('kagglehub', False),
    }
    for module, (requirement, no_deps) in packages.items():
        missing = importlib.util.find_spec(module) is None
        if module == 'freerec' and not missing:
            missing = importlib.metadata.version('freerec') != '0.9.7'
        if missing:
            command = [sys.executable, '-m', 'pip', 'install', '--quiet']
            if no_deps:
                command.append('--no-deps')
            subprocess.run(command + [requirement], check=True)

expected_sources = SEED1_REFERENCE['baby']['contract']['sources']
source_report = {}
for name, expected in expected_sources.items():
    choices = [REPO_ROOT / name, REPO_ROOT / 'models' / name, REPO_ROOT / 'optimizers' / name]
    path = next((p for p in choices if p.is_file()), None)
    if path is None:
        raise FileNotFoundError(f'Missing seed-1 source: {name}')
    payload = path.read_bytes()
    actual = hashlib.sha256(payload).hexdigest()
    lf_digest = hashlib.sha256(payload.replace(b'\r\n', b'\n')).hexdigest()
    source_report[name] = {'path': str(path.relative_to(REPO_ROOT)), 'sha256': actual,
                           'line_endings_only': actual != expected and lf_digest == expected}
    if expected not in (actual, lf_digest):
        raise RuntimeError(f'Algorithm source differs from seed 1: {path}. Use the pinned runtime commit.')
write_json(RUN_ROOT / 'provenance' / 'source_check.json', source_report)

probe = r"""
import json, platform, torch
import models.freerec_compat
import freerec
from main_stair_icaart import build_config
from stair_icaart_reporting import validate_selected_receipt
print('ECLIPSE_ENV=' + json.dumps({'python': platform.python_version(), 'torch': str(torch.__version__),
 'freerec': str(freerec.__version__), 'cuda': torch.cuda.is_available(),
 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))
"""
checked = subprocess.run([sys.executable, '-c', probe], cwd=REPO_ROOT, env=ENV,
                         text=True, encoding='utf-8', capture_output=True)
if checked.stdout:
    print(checked.stdout)
if checked.stderr:
    print(checked.stderr)
checked.check_returncode()
line = next(line for line in checked.stdout.splitlines() if line.startswith('ECLIPSE_ENV='))
RUNTIME_ENV = json.loads(line.split('=', 1)[1])
if not RUNTIME_ENV['cuda']:
    raise RuntimeError('Enable a Kaggle GPU accelerator before training.')
reference = SEED1_REFERENCE['baby']
deviations = {key: {'reference': reference[key], 'current': RUNTIME_ENV[key]}
              for key in ('torch', 'freerec') if RUNTIME_ENV[key] != reference[key]}
if RUNTIME_ENV['python'].split('.')[:2] != reference['python'].split('.')[:2]:
    deviations['python'] = {'reference': reference['python'], 'current': RUNTIME_ENV['python']}
write_json(RUN_ROOT / 'provenance' / 'environment.json', {'runtime': RUNTIME_ENV, 'deviations': deviations,
           'strict_runtime': STRICT_RUNTIME, 'gpu_index': GPU_INDEX})
if deviations and STRICT_RUNTIME:
    raise RuntimeError(f'Runtime differs from seed 1: {deviations}. Restore the seed-1 environment; '
                       'setting STRICT_RUNTIME=False explicitly records a protocol deviation.')
if deviations:
    print('PROTOCOL DEVIATION:', deviations)
if RUNTIME_ENV['gpu'] != reference['gpu']:
    print('HARDWARE DEVIATION:', RUNTIME_ENV['gpu'], 'vs', reference['gpu'],
          '- runtime measurements are not a matched-hardware comparison.')
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], text=True, capture_output=True, check=True)
(RUN_ROOT / 'provenance' / 'pip_freeze.txt').write_text(freeze.stdout, encoding='utf-8')
commit = subprocess.run(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True, capture_output=True)
ACTUAL_COMMIT = commit.stdout.strip() if commit.returncode == 0 else 'unavailable'
print('Runtime ready:', REPO_ROOT, '| commit:', ACTUAL_COMMIT)
print('Import/environment checks only; no model test or training has been executed.')


## 4. Chuẩn bị dữ liệu và kiểm tra fingerprints

Chỉ dùng dataset đã xử lý giống seed 1. Không tạo lại split hoặc features. Đích là thư mục writable `Processed/<dataset>` với file input liên kết từ Kaggle Input và thư mục `chunks` writable riêng. Có thể chỉ chọn Electronics để dành một GPU chạy seed 3, GPU/máy còn lại chạy notebook seed 2.

**Bắt buộc trước khi chạy batch trên Kaggle:**

1. Trong editor, chọn **Add Input** và gắn dataset `rainyle/stair-datasets-mmrec` hoặc processed data đã dùng ở seed 1. Dataset đã attach ở notebook khác không tự được attach vào notebook này.
2. Kiểm tra có các thư mục Baby/Sports/Electronics đã chọn; mỗi thư mục chứa `train.txt`, `valid.txt`, `test.txt`, `textual_modality.pkl`, `visual_modality.pkl`.
3. Sau khi attach, **Save Version / Save & Run All lại**. Một job batch đã khởi chạy không thể tự attach datasource mới.
4. Nếu dùng một cách sắp xếp thư mục khác, đặt `DATASET_SEARCH_ROOTS` hoặc `DATASET_SOURCE_OVERRIDES` ở cell cấu hình. Không trỏ tới file ZIP chưa giải nén hoặc chỉ tới output training thiếu input data.

Cell tìm dữ liệu trong repo/data, writable data bridge, Kaggle Input và cache KaggleHub hiện có; ghi `provenance/data_discovery.json` trước bước kiểm tra SHA-256. `AUTO_DOWNLOAD_DATA=True` chỉ cho phép fallback download ngoài Kaggle; notebook không gọi attach API trong Kaggle. Không tạo lại split/features và không bỏ qua fingerprint checks.

This is the w/o EC seed-3 run. The data-acquisition repair does not alter model configuration or fingerprints.


In [ ]:
import shutil

REQUIRED_FILES = ('train.txt', 'valid.txt', 'test.txt', 'textual_modality.pkl', 'visual_modality.pkl')


def dataset_search_roots(extra_roots=()):
    # Include writable bridges and caches created by earlier interactive sessions.
    return [REPO_ROOT / 'data', DATA_ROOT, WORK_DIR / 'data',
            Path('/kaggle/data'), Path('/kaggle/input'),
            Path.home() / '.cache' / 'kagglehub' / 'datasets',
            *map(Path, DATASET_SEARCH_ROOTS), *map(Path, extra_roots)]


def discover_datasets(extra_roots=()):
    found = {}
    unknown = set(DATASET_SOURCE_OVERRIDES) - set(DATASET_FOLDERS)
    if unknown:
        raise ValueError(f'Unknown DATASET_SOURCE_OVERRIDES keys: {sorted(unknown)}')
    for key in SELECTED_DATASETS:
        if key not in DATASET_SOURCE_OVERRIDES:
            continue
        path = Path(DATASET_SOURCE_OVERRIDES[key]).expanduser()
        absent = [name for name in REQUIRED_FILES if not (path / name).is_file()]
        if absent:
            raise FileNotFoundError(f'Incomplete explicit dataset source for {key}: {path}; missing {absent}')
        found[key] = path
    aliases = {'baby': ('baby',), 'sports': ('sport',), 'electronics': ('electronic',)}
    visited = set()
    for root in dataset_search_roots(extra_roots):
        root = root.expanduser()
        if not root.is_dir() or str(root.resolve()) in visited:
            continue
        visited.add(str(root.resolve()))
        for current, directories, files in os.walk(root, followlinks=False):
            directories[:] = sorted(d for d in directories if d not in
                                    {'.git', 'chunks', '__pycache__', '.stair_icaart_graph_cache'})
            if not set(REQUIRED_FILES).issubset(files):
                continue
            path = Path(current)
            for key in SELECTED_DATASETS:
                if any(alias in path.name.lower() for alias in aliases[key]):
                    found.setdefault(key, path)
    return found


def missing_dataset_message(missing):
    return (
        f'Missing processed datasets: {sorted(missing)}.\n'
        f'On Kaggle, attach {DATASET_HANDLE!r} with Add Input BEFORE Save & Run All '
        '(or attach an equivalent processed dataset matching the seed-1 fingerprints). '
        'Save a new notebook version after attaching it. Internet access alone does not '
        'attach a datasource, and kagglehub cannot attach a new one in a non-interactive run.\n'
        'Alternatively set DATASET_SOURCE_OVERRIDES to the existing per-dataset folders, '
        'or DATASET_SEARCH_ROOTS to their parent directory. Each dataset folder must contain: '
        + ', '.join(REQUIRED_FILES) + '.\n'
        'Searched roots: ' + ', '.join(str(p.expanduser()) for p in dataset_search_roots())
        + f'\nDiscovery receipt: {RUN_ROOT / "provenance" / "data_discovery.json"}'
    )


def resolve_dataset_sources():
    sources = discover_datasets()
    missing = set(SELECTED_DATASETS) - sources.keys()
    on_kaggle = Path('/kaggle/working').is_dir()
    discovery = {'dataset_handle': DATASET_HANDLE, 'selected_datasets': list(SELECTED_DATASETS),
                 'searched_roots': [str(p.expanduser()) for p in dataset_search_roots()],
                 'on_kaggle': on_kaggle, 'auto_download_requested': AUTO_DOWNLOAD_DATA,
                 'download_attempted': False}

    def record(status):
        discovery.update(status=status, missing=sorted(missing),
                         sources={key: str(path.resolve()) for key, path in sources.items()})
        write_json(RUN_ROOT / 'provenance' / 'data_discovery.json', discovery)

    if missing and on_kaggle:
        # Never try to attach a datasource during a Kaggle batch job. The caller
        # must attach inputs before execution; this also makes batch runs repeatable.
        record('MISSING_ATTACHED_INPUT')
        raise FileNotFoundError(missing_dataset_message(missing))
    if missing and AUTO_DOWNLOAD_DATA:
        discovery['download_attempted'] = True
        try:
            import kagglehub
            downloaded = kagglehub.dataset_download(DATASET_HANDLE)
            discovery['downloaded_root'] = str(downloaded)
            sources.update(discover_datasets([downloaded]))
            missing = set(SELECTED_DATASETS) - sources.keys()
        except Exception as exc:
            discovery['download_error'] = {'type': type(exc).__name__, 'message': str(exc)}
            record('DOWNLOAD_FAILED')
            raise FileNotFoundError(missing_dataset_message(missing)) from exc
    if missing:
        record('MISSING_PROCESSED_DATA')
        raise FileNotFoundError(missing_dataset_message(missing))
    record('FOUND_PENDING_FINGERPRINT_CHECK')
    return sources


sources = resolve_dataset_sources()

DATA_RECEIPTS = {}
for key in SELECTED_DATASETS:
    source = sources[key].resolve()
    reference = SEED1_REFERENCE[key]
    expected = dict(reference['contract']['splits'])
    expected.update({record['file']: record['sha256'] for record in reference['features']})
    destination = DATA_ROOT / 'Processed' / DATASET_FOLDERS[key]
    if destination.is_symlink():
        raise RuntimeError(f'Processed destination must be a writable directory, not a symlink: {destination}')
    destination.mkdir(parents=True, exist_ok=True)
    for name, expected_hash in expected.items():
        original = source / name
        if not original.is_file():
            raise FileNotFoundError(original)
        actual_hash = file_sha256(original)
        if actual_hash != expected_hash:
            raise RuntimeError(f'Dataset differs from seed 1: {original}')
        target = destination / name
        if target.is_symlink() and not target.exists():
            raise FileExistsError(f'Broken input link preserved; choose a fresh DATA_ROOT: {target}')
        if target.exists():
            if file_sha256(target) != expected_hash:
                raise RuntimeError(f'Existing linked/copied input has a different fingerprint: {target}')
        else:
            try:
                target.symlink_to(original)
            except OSError:
                shutil.copy2(original, target)
    if (destination / 'chunks').is_symlink():
        raise RuntimeError(f'FreeRec chunks must be writable, not an input symlink: {destination / "chunks"}')
    (destination / 'chunks').mkdir(exist_ok=True)
    DATA_RECEIPTS[key] = {'source': str(source), 'processed': str(destination), 'fingerprints': expected}
    print(f'{DATASET_TITLES[key]}: split and feature fingerprints match seed 1.')
write_json(RUN_ROOT / 'provenance' / 'data_check.json', DATA_RECEIPTS)


## 5. Cấu hình đã khóa và pre-flight CLI

Các YAML gốc không bị sửa. Notebook tạo bản YAML riêng theo protocol thực tế seed 1 rồi đổi seed huấn luyện. `evidence_seed` không hoạt động ở pooled-full và không được đổi thành training seed. Không thay các tham số delta/tau, batch size, worker count, chunk size hoặc evaluation frequency để tìm kết quả đẹp hơn.

Cell này chỉ kiểm tra khả năng import và cấu hình đã resolve; **không chạy unit tests, không build graph, không train**. FreeRec có thể tạo thư mục log rỗng trong bước compile config.


In [ ]:
import yaml

CONFIG_PATHS = {}
ARTIFACT_PATHS = {}
LOG_PATHS = {}
for key in SELECTED_DATASETS:
    title = key.capitalize()
    original = REPO_ROOT / 'configs' / f'Amazon2014{title}_STAIR_PEEC.yaml'
    if not original.is_file():
        raise FileNotFoundError(original)
    config = yaml.safe_load(original.read_text(encoding='utf-8'))
    config.update(SEED1_REFERENCE[key]['v8_config'])
    config.update(SEED1_REFERENCE[key]['icaart_config'])
    config['evidence_delta'] = 0.0
    config.pop('evidence_seed', None)  # Inactive; do not confuse the evidence partition with training RNG.
    config.update(seed=SEED, root=str(DATA_ROOT), dataset=DATASET_FOLDERS[key],
                  num_workers=SEED1_REFERENCE[key]['contract']['config']['num_workers'],
                  eval_valid=True, eval_test=False, ranking='full', which4best='NDCG@20',
                  resume=False, resume_from='')
    config['mfiles'] = ','.join(config['mfiles'])
    config['num_neighbors'] = '-'.join(map(str, config['num_neighbors']))
    artifact = RUN_ROOT / 'artifacts' / f'{key}_eclipse_wo_ec_seed{SEED}'
    config['artifact_dir'] = str(artifact)
    config['graph_cache_dir'] = str(GRAPH_CACHE_ROOT / key)
    target = RUN_ROOT / 'configs' / f'{key}_eclipse_wo_ec_seed{SEED}.yaml'
    target.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
    CONFIG_PATHS[key], ARTIFACT_PATHS[key] = target, artifact
    LOG_PATHS[key] = RUN_ROOT / 'logs' / f'{key}_eclipse_wo_ec_seed{SEED}.log'


def build_command(key):
    if key not in SELECTED_DATASETS:
        raise ValueError(f'{key} is not selected; edit SELECTED_DATASETS and rerun setup.')
    return [sys.executable, '-u', str(REPO_ROOT / 'main_stair_icaart.py'),
            '--config', str(CONFIG_PATHS[key]), '--seed', str(SEED),
            '--root', str(DATA_ROOT), '--dataset', DATASET_FOLDERS[key],
            '--device', '0', '--ranking', 'full',
            '--artifact-dir', str(ARTIFACT_PATHS[key]),
            '--graph-cache-dir', str(GRAPH_CACHE_ROOT / key)]


config_probe = r"""
import json, sys
from main_stair_icaart import build_config
cfg = build_config(sys.argv[1:])
keys = json.loads(__import__('os').environ['ECLIPSE_CHECK_KEYS'])
receipt = {key: getattr(cfg, key) for key in keys}
print('ECLIPSE_CONFIG=' + json.dumps(receipt))
"""
for key in SELECTED_DATASETS:
    expected = dict(SEED1_REFERENCE[key]['v8_config'])
    expected.update(SEED1_REFERENCE[key]['icaart_config'])
    expected['evidence_delta'] = 0.0
    expected.update(seed=SEED, dataset=DATASET_FOLDERS[key], ranking='full', which4best='NDCG@20',
                    eval_valid=True, eval_test=False, resume=False, resume_from='',
                    num_workers=SEED1_REFERENCE[key]['contract']['config']['num_workers'])
    probe_env = ENV.copy()
    probe_env['ECLIPSE_CHECK_KEYS'] = json.dumps(list(expected))
    result = subprocess.run([sys.executable, '-c', config_probe, *build_command(key)[3:]],
                            cwd=REPO_ROOT, env=probe_env, text=True, encoding='utf-8', capture_output=True)
    if result.returncode:
        print(result.stdout, result.stderr)
        result.check_returncode()
    payload = next(line.split('=', 1)[1] for line in result.stdout.splitlines()
                   if line.startswith('ECLIPSE_CONFIG='))
    resolved = json.loads(payload)
    changes = {name: {'expected': value, 'actual': resolved.get(name)}
               for name, value in expected.items() if resolved.get(name) != value}
    if changes:
        raise RuntimeError(f'{key}: resolved CLI/YAML protocol mismatch: {changes}')
    write_json(RUN_ROOT / 'provenance' / f'{key}_resolved_config.json', resolved)
    print(f'{key}: seed={resolved["seed"]}, epochs={resolved["epochs"]}, '
          f'batch={resolved["batch_size"]}, gamma={resolved["gamma"]}, '
          f'weight_decay={resolved["weight_decay"]}, cl_chunk={resolved["cl_chunk_size"]}')
write_json(RUN_ROOT / 'run_manifest.json', {
    'method': 'ECLIPSE', 'arm': 'wo_ec', 'training_seed': SEED,
    'datasets': list(SELECTED_DATASETS), 'session_id': SESSION_ID,
    'reference_commit': RUNTIME_COMMIT, 'actual_commit': ACTUAL_COMMIT,
    'runtime': RUNTIME_ENV, 'runtime_deviations': deviations,
    'selection': 'validation NDCG@20', 'test_policy': 'selected checkpoint only',
    'configs': {k: {'path': str(p), 'sha256': file_sha256(p)} for k, p in CONFIG_PATHS.items()},
    'dataset_paths': DATA_RECEIPTS,
    'note': 'w/o EC only: evidence_delta=0.0; no resume from seed 1.',
})


## 6. Training engine và dữ liệu xuất bản

Runner dùng chính `main_stair_icaart.py`; notebook chỉ orchestration và đọc receipts. Checkpoints khác seed không được nạp. Khi phát hiện candidate/operator/protocol mismatch, subprocess riêng của notebook bị dừng và log được giữ lại. Rerun cell của một run hoàn tất chỉ đọc lại kết quả; run dở không bị xóa hoặc tự retry.

`selected_metrics.json` là nguồn TEST duy nhất. Learning curves dùng `evaluation_metrics.jsonl` mode VALID; allocator curves dùng telemetry của **child training process**, không dùng bộ nhớ kernel notebook hay NVML toàn GPU để gắn nhãn tensor memory.


In [ ]:
import math
import re
import signal
import time
import pandas as pd
import numpy as np
from IPython.display import display

RESULTS = {}


def read_jsonl(path):
    path = Path(path)
    if not path.is_file():
        return []
    result = []
    with path.open(encoding='utf-8') as handle:
        for number, line in enumerate(handle, 1):
            if line.strip():
                try:
                    result.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f'Malformed JSONL: {path}, line {number}') from exc
    return result


def compare_manifest(key, manifest, expected_seed):
    reference = SEED1_REFERENCE[key]
    problems = {}
    if manifest.get('seed') != expected_seed:
        problems['seed'] = manifest.get('seed')
    if manifest.get('selection') != 'validation NDCG@20':
        problems['selection'] = manifest.get('selection')
    for name, expected in reference['v8_config'].items():
        actual = manifest.get('v8_config', {}).get(name)
        if actual != expected:
            problems[f'config.{name}'] = {'expected': expected, 'actual': actual}
    expected_evidence = dict(reference['icaart_config'], evidence_delta=0.0)
    if manifest.get('icaart_config') != expected_evidence:
        problems['evidence_config'] = manifest.get('icaart_config')
    contract = manifest.get('checkpoint_contract', {})
    for name in ('splits', 'candidate_hashes', 'data_fingerprint', 'graph_fingerprint'):
        expected = WO_EC_GRAPH[key] if name == 'graph_fingerprint' else reference['contract'][name]
        if contract.get(name) != expected:
            problems[name] = {'expected': expected, 'actual': contract.get(name)}
    for name, expected in reference['contract']['config'].items():
        if name == 'seed':
            expected = expected_seed
        if contract.get('config', {}).get(name) != expected:
            problems[f'checkpoint_config.{name}'] = contract.get('config', {}).get(name)
    if manifest.get('features') != reference['features']:
        problems['features'] = manifest.get('features')
    if manifest.get('precision') != {'tf32': False, 'amp': False}:
        problems['precision'] = manifest.get('precision')
    diagnostic = manifest.get('graph', {}).get('icaart', {})
    if (diagnostic.get('exact_v8_bypass') is not True
            or diagnostic.get('inactive_reason') != 'delta_zero'
            or diagnostic.get('relative_operator_dose') != 0.0
            or diagnostic.get('relative_polynomial_dose') != 0.0):
        problems['delta_zero_bypass'] = diagnostic
    if problems:
        raise RuntimeError(f'{key}: seed-1 comparability check failed: {problems}')
    return {'status': 'MATCH', 'reference_seed': 1, 'training_seed': expected_seed,
            'same_candidates': True, 'same_graph_operator': True,
            'same_splits_features': True, 'same_model_configuration_except_seed': True}


def validate_receipt(key, receipt, manifest, expected_seed):
    if (receipt.get('mode') != 'test' or receipt.get('selected_checkpoint') is not True
            or receipt.get('selection') != 'validation NDCG@20'
            or receipt.get('seed') != expected_seed
            or receipt.get('evidence_mode') != 'pooled_activity_excess'
            or receipt.get('evidence_delta') != 0.0):
        raise ValueError(f'{key}: not a selected-checkpoint ECLIPSE w/o EC TEST receipt')
    if receipt.get('graph_fingerprint') != manifest.get('checkpoint_contract', {}).get('graph_fingerprint'):
        raise ValueError(f'{key}: metric receipt belongs to another operator')
    epoch = receipt.get('epoch')
    if isinstance(epoch, bool) or not isinstance(epoch, int) or not 0 <= epoch <= 500:
        raise ValueError(f'{key}: invalid selected epoch')
    normalized = {str(k).upper(): v for k, v in receipt.get('metrics', {}).items()}
    for metric in PRIMARY_METRICS:
        value = normalized.get(metric)
        if isinstance(value, bool) or not isinstance(value, (float, int)) or not math.isfinite(value) or not 0 <= value <= 1:
            raise ValueError(f'{key}: missing/invalid measured TEST metric {metric}')
    return normalized


def export_run_tables(key):
    artifact = ARTIFACT_PATHS[key]
    manifest = json.loads((artifact / 'manifest.json').read_text(encoding='utf-8'))
    status = json.loads((artifact / 'run_status.json').read_text(encoding='utf-8'))
    if status.get('status') != 'COMPLETED' or status.get('seed') != SEED:
        raise RuntimeError(f'{key}: run is not completed for training seed {SEED}')
    comparability = compare_manifest(key, manifest, SEED)
    receipt = json.loads((artifact / 'selected_metrics.json').read_text(encoding='utf-8'))
    metrics = validate_receipt(key, receipt, manifest, SEED)
    telemetry = pd.DataFrame(read_jsonl(artifact / 'training_telemetry.jsonl'))
    evaluations = read_jsonl(artifact / 'evaluation_metrics.jsonl')
    if telemetry.empty or int(telemetry['epoch'].max()) != 500 or telemetry['epoch'].duplicated().any():
        raise RuntimeError(f'{key}: expected one uninterrupted 500-epoch attempt')
    telemetry.to_csv(RUN_ROOT / 'tables' / f'eclipse_wo_ec_{key}_seed{SEED}_training_telemetry.csv', index=False, na_rep='NOT_MEASURED')
    evaluation_rows = [{k: v for k, v in record.items() if k != 'metrics'} |
                       {str(k).upper(): v for k, v in record.get('metrics', {}).items()}
                       for record in evaluations]
    pd.DataFrame(evaluation_rows).to_csv(RUN_ROOT / 'tables' / f'eclipse_wo_ec_{key}_seed{SEED}_evaluations.csv',
                                       index=False, na_rep='NOT_MEASURED')
    log_text = LOG_PATHS[key].read_text(encoding='utf-8', errors='replace') if LOG_PATHS[key].is_file() else ''
    fit_matches = re.findall(r'Coach\.fit takes\s+([0-9.eE+-]+)\s+seconds', log_text)
    summary = {'method': 'ECLIPSE', 'arm': 'wo_ec', 'dataset': key, 'seed': SEED,
               'selected_epoch': receipt['epoch'], 'split': 'TEST', **metrics,
               'fit_seconds': float(fit_matches[-1]) if fit_matches else None,
               'train_epoch_seconds_sum': float(telemetry['seconds'].sum()),
               'preprocessing_seconds': manifest.get('preprocessing_seconds'),
               'peak_training_allocated_mib': float(telemetry['peak_allocated_gib'].max() * 1024),
               'peak_training_reserved_mib': float(telemetry['peak_reserved_gib'].max() * 1024),
               'memory_scope': 'child-process per-training-epoch CUDA allocator peaks; excludes evaluation peaks',
               'commit': manifest.get('commit'), 'graph_fingerprint': receipt['graph_fingerprint'],
               'artifact_dir': str(artifact), 'checkpoint_path': str(artifact / 'best_model.pt'),
               'runtime_deviations': deviations, 'comparability': comparability}
    write_json(RUN_ROOT / 'tables' / f'eclipse_wo_ec_{key}_seed{SEED}_selected_test.json', summary)
    write_json(RUN_ROOT / 'tables' / f'eclipse_wo_ec_{key}_seed{SEED}_mechanism.json', manifest['graph']['icaart'])
    RESULTS[key] = summary
    return summary


def train_dataset(key):
    if key not in SELECTED_DATASETS:
        print(f'Skipped {key}: not in SELECTED_DATASETS.')
        return None
    artifact = ARTIFACT_PATHS[key]
    status_path = artifact / 'run_status.json'
    if status_path.is_file():
        status = json.loads(status_path.read_text(encoding='utf-8'))
        if status.get('status') == 'COMPLETED':
            print(f'{key}: reading the completed attempt; no retraining.')
            return export_run_tables(key)
    if (artifact.is_dir() and any(artifact.iterdir())) or LOG_PATHS[key].exists():
        raise FileExistsError(f'Incomplete/existing attempt preserved at {artifact}; '
                              'rerun setup for a new SESSION_ID. No automatic resume/delete/retry.')
    artifact.mkdir(parents=True, exist_ok=True)
    command = build_command(key)
    write_json(RUN_ROOT / 'provenance' / f'{key}_command.json', {
        'argv': command, 'training_seed': SEED, 'pythonhashseed': ENV['PYTHONHASHSEED'],
        'cuda_visible_devices': ENV['CUDA_VISIBLE_DEVICES'], 'cwd': str(REPO_ROOT)})
    print('Running:', ' '.join(command))
    start = time.perf_counter()
    manifest_checked = False
    process = None
    try:
        with LOG_PATHS[key].open('x', encoding='utf-8') as log:
            process = subprocess.Popen(command, cwd=REPO_ROOT, env=ENV, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                                       errors='replace', bufsize=1)
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end='', flush=True)
                manifest_path = artifact / 'manifest.json'
                if not manifest_checked and manifest_path.is_file():
                    comparability = compare_manifest(key, json.loads(manifest_path.read_text(encoding='utf-8')), SEED)
                    write_json(RUN_ROOT / 'provenance' / f'{key}_comparability.json', comparability)
                    manifest_checked = True
            process.wait()
            process.stdout.close()
        if process.returncode != 0:
            raise RuntimeError(f'{key}: training failed with exit code {process.returncode}; see {LOG_PATHS[key]}')
        result = export_run_tables(key)
        write_json(RUN_ROOT / 'provenance' / f'{key}_notebook_execution.json', {
            'status': 'COMPLETED', 'subprocess_wall_seconds': time.perf_counter() - start,
            'exit_code': process.returncode, 'training_seed': SEED})
        print(f'\nCompleted {DATASET_TITLES[key]} | selected TEST NDCG@20={result["NDCG@20"]:.10f}')
        return result
    except BaseException as exc:
        if process is not None and process.poll() is None:
            if isinstance(exc, KeyboardInterrupt):
                process.send_signal(signal.SIGINT)
            else:
                process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=10)
        write_json(RUN_ROOT / 'provenance' / f'{key}_notebook_execution.json', {
            'status': 'INTERRUPTED' if isinstance(exc, KeyboardInterrupt) else 'FAILED',
            'error_type': type(exc).__name__, 'error': str(exc), 'training_seed': SEED,
            'subprocess_wall_seconds': time.perf_counter() - start})
        raise


## 7. Figure utilities — PDF vector, PNG 300 dpi, dữ liệu nguồn

Các biểu đồ là mô tả run hiện tại, không phải bằng chứng significance. Không hard-code số thứ tự Figure 1/2/3: tên file mô tả nội dung và seed để có thể đổi bố cục paper sau này.

- `eclipse_<dataset>_seed3_learning_curves.{pdf,png}`: loss và VALID ranking metrics; đánh dấu epoch chọn checkpoint.
- `eclipse_<dataset>_seed3_training_memory.{pdf,png}`: allocator peaks của training subprocess theo epoch.
- `eclipse_<dataset>_seed3_calibration_diagnostics.{pdf,png}`: coverage, operator/polynomial dose và score quantiles thực đo.
- Không tạo histogram giả từ quantiles; không vẽ TEST theo epoch để chọn mô hình.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.titlesize': 10,
                     'axes.labelsize': 9, 'legend.fontsize': 8, 'xtick.labelsize': 8,
                     'ytick.labelsize': 8, 'lines.linewidth': 1.5,
                     'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.dpi': 300,
                     'axes.spines.top': False, 'axes.spines.right': False})
COLORS = {'blue': '#0072B2', 'orange': '#D55E00', 'green': '#009E73',
          'purple': '#CC79A7', 'gray': '#666666'}
_existing_figure_manifest = RUN_ROOT / 'figures' / 'figure_manifest.json'
FIGURE_RECORDS = json.loads(_existing_figure_manifest.read_text(encoding='utf-8')) if _existing_figure_manifest.is_file() else []


def save_figure(fig, stem, caption, sources):
    for extension in ('pdf', 'png'):
        fig.savefig(RUN_ROOT / 'figures' / f'{stem}.{extension}', bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
    FIGURE_RECORDS[:] = [record for record in FIGURE_RECORDS if record['stem'] != stem]
    FIGURE_RECORDS.append({'stem': stem, 'caption': caption, 'sources': list(map(str, sources))})
    write_json(RUN_ROOT / 'figures' / 'figure_manifest.json', FIGURE_RECORDS)
    print(caption)


def completed_artifact(key):
    if key not in SELECTED_DATASETS:
        print(f'{key}: not selected; no figure generated.')
        return None
    path = ARTIFACT_PATHS[key]
    status = path / 'run_status.json'
    if not status.is_file() or json.loads(status.read_text(encoding='utf-8')).get('status') != 'COMPLETED':
        print(f'{key}: no completed run available; no figure generated.')
        return None
    return path


def plot_learning_curves(key):
    artifact = completed_artifact(key)
    if artifact is None:
        return
    telemetry = pd.DataFrame(read_jsonl(artifact / 'training_telemetry.jsonl'))
    records = [record for record in read_jsonl(artifact / 'evaluation_metrics.jsonl')
               if record.get('mode') == 'valid' and not record.get('selected_checkpoint')]
    selected = json.loads((artifact / 'selected_metrics.json').read_text(encoding='utf-8'))['epoch']
    valid = pd.DataFrame([{'epoch': record['epoch'],
                           **{str(k).upper(): v for k, v in record['metrics'].items()}}
                          for record in records]).sort_values('epoch')
    if telemetry.empty or valid.empty:
        print(f'{key}: missing measured curves; no substitute data generated.')
        return
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0), constrained_layout=True)
    for column, label, color in [('loss', 'Total', COLORS['blue']),
                                  ('bpr_loss', 'BPR', COLORS['orange']),
                                  ('weighted_nlgcl_loss', 'Weighted LC', COLORS['green'])]:
        if column in telemetry:
            axes[0].plot(telemetry['epoch'], telemetry[column], label=label, color=color)
    axes[0].set(xlabel='Training epoch', ylabel='Mean training loss', title='Training objective')
    axes[0].legend(frameon=False)
    for ax, metric in zip(axes[1:], ('NDCG@20', 'RECALL@20')):
        if metric in valid:
            ax.plot(valid['epoch'], valid[metric], color=COLORS['blue'], label=f'VALID {metric}')
            point = valid.loc[valid['epoch'] == selected, metric]
            if len(point):
                ax.scatter([selected], [point.iloc[-1]], color=COLORS['orange'], s=22, zorder=3)
        ax.axvline(selected, color=COLORS['gray'], linestyle='--', linewidth=1, label=f'Selected epoch {selected}')
        ax.set(xlabel='Training epoch', ylabel=f'Validation {metric}', title=f'Validation {metric}')
        ax.legend(frameon=False)
    for ax in axes:
        ax.grid(alpha=.2)
    fig.suptitle(f'ECLIPSE w/o EC | {DATASET_TITLES[key]} | seed {SEED}', fontsize=11)
    save_figure(fig, f'eclipse_wo_ec_{key}_seed{SEED}_learning_curves',
                'Training objective and validation ranking curves. The checkpoint is selected by validation NDCG@20; '
                'TEST is evaluated at the selected checkpoint and is not a learning-curve selection signal.',
                [artifact / 'training_telemetry.jsonl', artifact / 'evaluation_metrics.jsonl', artifact / 'selected_metrics.json'])


def plot_training_memory(key):
    artifact = completed_artifact(key)
    if artifact is None:
        return
    telemetry = pd.DataFrame(read_jsonl(artifact / 'training_telemetry.jsonl'))
    if telemetry.empty:
        return
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), constrained_layout=True)
    for column, label, color in [('peak_allocated_gib', 'Peak allocated', COLORS['blue']),
                                  ('peak_reserved_gib', 'Peak reserved', COLORS['orange'])]:
        if column in telemetry:
            axes[0].plot(telemetry['epoch'], telemetry[column] * 1024, label=label, color=color)
    axes[0].set(xlabel='Training epoch', ylabel='CUDA allocator memory (MiB)', title='Training-epoch allocator peaks')
    axes[0].legend(frameon=False)
    axes[1].plot(telemetry['epoch'], telemetry['seconds'], color=COLORS['green'])
    axes[1].set(xlabel='Training epoch', ylabel='Training time (s/epoch)', title='Training time per epoch')
    for ax in axes:
        ax.grid(alpha=.2)
    fig.suptitle(f'ECLIPSE w/o EC | {DATASET_TITLES[key]} | seed {SEED}', fontsize=11)
    save_figure(fig, f'eclipse_wo_ec_{key}_seed{SEED}_training_memory',
                'CUDA allocator peaks are measured in the training subprocess for each training epoch. '
                'They exclude evaluation peaks, other GPU processes and driver memory. No arbitrary hardware ceiling is drawn.',
                [artifact / 'training_telemetry.jsonl'])


def plot_calibration_diagnostics(key):
    artifact = completed_artifact(key)
    if artifact is None:
        return
    manifest = json.loads((artifact / 'manifest.json').read_text(encoding='utf-8'))
    diagnostics = manifest['graph']['icaart']
    coverage = [diagnostics.get('frac_edges_with_positive_evidence'), diagnostics.get('weighted_evidence_coverage')]
    if any(value is None for value in coverage):
        print(f'{key}: coverage was not measured.')
        return
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0), constrained_layout=True)
    bars = axes[0].bar(['Candidate\nedges', 'Raw semantic\nweight'], np.asarray(coverage) * 100,
                       color=[COLORS['blue'], COLORS['green']], width=.6)
    axes[0].bar_label(bars, fmt='%.2f%%', padding=3, fontsize=8)
    axes[0].set(ylabel='Coverage with positive evidence (%)', title='Calibration coverage')
    axes[0].set_ylim(0, max(coverage) * 125 + 1)
    polynomial = diagnostics.get('polynomial_dose_by_coordinate', {})
    if polynomial:
        coordinates = sorted(map(int, polynomial))
        values = [100 * polynomial[str(coordinate)] for coordinate in coordinates]
        axes[1].plot(coordinates, values, 'o-', color=COLORS['blue'], label='Polynomial probe dose')
    operator_dose = diagnostics.get('relative_operator_dose')
    if operator_dose is not None:
        axes[1].axhline(100 * operator_dose, color=COLORS['orange'], linestyle='--', label='Operator Frobenius dose')
    axes[1].set(xlabel='Embedding coordinate', ylabel='Relative dose (%)', title='Operator / polynomial\nprobe dose')
    axes[1].legend(frameon=False)
    levels, values = diagnostics.get('evidence_quantile_levels'), diagnostics.get('evidence_score_quantiles')
    if levels is not None and values is not None and len(levels) == len(values):
        axes[2].plot(np.asarray(levels) * 100, values, 'o-', color=COLORS['green'])
        pd.DataFrame({'quantile': levels, 'evidence_score': values}).to_csv(
            RUN_ROOT / 'tables' / f'eclipse_wo_ec_{key}_seed{SEED}_evidence_quantiles.csv', index=False)
    else:
        axes[2].text(.5, .5, 'Score quantiles not measured', ha='center', va='center', transform=axes[2].transAxes)
    axes[2].set(xlabel='Quantile (%)', ylabel='Excess-evidence score', ylim=(0, 1), title='Evidence-score quantile profile')
    for ax in axes:
        ax.grid(axis='y', alpha=.2)
    fig.suptitle(f'ECLIPSE w/o EC | {DATASET_TITLES[key]} | seed {SEED}', fontsize=11)
    save_figure(fig, f'eclipse_wo_ec_{key}_seed{SEED}_calibration_diagnostics',
                'w/o EC: evidence may be present, but operator and polynomial calibration dose must be exactly zero. '
                'Polynomial dose uses fixed Rademacher probes, not actual Adam directions. '
                'The score panel displays measured quantiles, not a reconstructed histogram.', [artifact / 'manifest.json'])


def plot_architecture():
    fig, ax = plt.subplots(figsize=(7.2, 2.7))
    ax.axis('off')
    blocks = [('Text / image\nWMSG candidates + weights', COLORS['blue']),
              ('EC BYPASS\nevidence delta = 0', COLORS['orange']),
              ('Semantic / CF blend\neta = 0.1', COLORS['green']),
              ('Item-BSC on Adam direction\nMI/FSC + BPR + LC retained', COLORS['blue'])]
    for i, (label, color) in enumerate(blocks):
        x = .02 + i * .25
        ax.add_patch(FancyBboxPatch((x, .35), .21, .38, boxstyle='round,pad=0.01',
                     edgecolor=color, facecolor=color + '14', transform=ax.transAxes))
        ax.text(x + .105, .54, label, ha='center', va='center', fontsize=8, transform=ax.transAxes)
        if i < 3:
            ax.annotate('', xy=(x+.25,.54), xytext=(x+.21,.54), xycoords='axes fraction',
                        arrowprops={'arrowstyle':'->'})
    ax.set_title('ECLIPSE w/o EC: only calibration is disabled')
    save_figure(fig, 'eclipse_wo_ec_architecture',
        'The ablation keeps semantic weighting, collaborative support expansion, layer-contrastive learning '
        'and optimizer-side BSC. Delta zero routes to the exact uncalibrated operator. '
        'Evidence statistics can still be computed offline, but the calibration operator dose is zero.', [])


## 8. Architecture figure (không cần chạy training)

Bản vector này là sơ đồ khởi đầu cho paper; kiểm tra bố cục sau khi xuất. Các figure accuracy/ablation so sánh phương pháp khác chỉ được tạo sau khi có dữ liệu đo tương ứng.


In [ ]:
plot_architecture()


## Training — Amazon Sports, ECLIPSE w/o EC

Cell này train từ đầu với `SEED`, không đổi cấu hình theo dataset ngoài các hyperparameters seed 1 đã khóa. Nếu dataset không nằm trong `SELECTED_DATASETS`, cell được bỏ qua rõ ràng.


In [ ]:
sports_result = train_dataset('sports')
if sports_result is not None:
    display(pd.DataFrame([sports_result])[["dataset", "seed", "selected_epoch", *PRIMARY_METRICS]])


### Amazon Sports — validation learning curves


In [ ]:
plot_learning_curves('sports')


### Amazon Sports — training allocator memory và thời gian


In [ ]:
plot_training_memory('sports')


### Amazon Sports — calibration diagnostics


In [ ]:
plot_calibration_diagnostics('sports')


## Training — Amazon Baby, ECLIPSE w/o EC

Cell này train từ đầu với `SEED`, không đổi cấu hình theo dataset ngoài các hyperparameters seed 1 đã khóa. Nếu dataset không nằm trong `SELECTED_DATASETS`, cell được bỏ qua rõ ràng.


In [ ]:
baby_result = train_dataset('baby')
if baby_result is not None:
    display(pd.DataFrame([baby_result])[["dataset", "seed", "selected_epoch", *PRIMARY_METRICS]])


### Amazon Baby — validation learning curves


In [ ]:
plot_learning_curves('baby')


### Amazon Baby — training allocator memory và thời gian


In [ ]:
plot_training_memory('baby')


### Amazon Baby — calibration diagnostics


In [ ]:
plot_calibration_diagnostics('baby')


## Training — Amazon Electronics, ECLIPSE w/o EC

Cell này train từ đầu với `SEED`, không đổi cấu hình theo dataset ngoài các hyperparameters seed 1 đã khóa. Nếu dataset không nằm trong `SELECTED_DATASETS`, cell được bỏ qua rõ ràng.


In [ ]:
electronics_result = train_dataset('electronics')
if electronics_result is not None:
    display(pd.DataFrame([electronics_result])[["dataset", "seed", "selected_epoch", *PRIMARY_METRICS]])


### Amazon Electronics — validation learning curves


In [ ]:
plot_learning_curves('electronics')


### Amazon Electronics — training allocator memory và thời gian


In [ ]:
plot_training_memory('electronics')


### Amazon Electronics — calibration diagnostics


In [ ]:
plot_calibration_diagnostics('electronics')


## 9. Selected TEST table và resource profile

Chỉ ghi kết quả đo của seed hiện tại; thiếu metric là `NOT_MEASURED`. CSV giữ độ chính xác số thực. Không gắn dấu significance hoặc so với scalar baseline làm tròn. Resource profile của một phương pháp không phải bằng chứng accuracy-efficiency superiority.


In [ ]:
summary_rows = []
for key in SELECTED_DATASETS:
    if completed_artifact(key) is None:
        summary_rows.append({'method': 'ECLIPSE', 'dataset': key, 'seed': SEED, 'status': 'NOT_MEASURED'})
        continue
    summary = export_run_tables(key)
    names = ['method', 'arm', 'dataset', 'seed', 'selected_epoch', *PRIMARY_METRICS,
             'preprocessing_seconds', 'fit_seconds', 'train_epoch_seconds_sum',
             'peak_training_allocated_mib', 'peak_training_reserved_mib',
             'checkpoint_path', 'commit', 'graph_fingerprint']
    summary_rows.append({name: summary.get(name) for name in names} | {'status': 'COMPLETED'})
SUMMARY = pd.DataFrame(summary_rows)
for column in ['selected_epoch', *PRIMARY_METRICS, 'preprocessing_seconds', 'fit_seconds',
               'peak_training_allocated_mib', 'peak_training_reserved_mib']:
    if column not in SUMMARY:
        SUMMARY[column] = np.nan
SUMMARY.to_csv(RUN_ROOT / 'tables' / f'eclipse_wo_ec_seed{SEED}_selected_test.csv', index=False, na_rep='NOT_MEASURED')
display(SUMMARY[['dataset', 'seed', 'status', 'selected_epoch', *PRIMARY_METRICS]].fillna('NOT_MEASURED'))
measured = SUMMARY.loc[SUMMARY['status'] == 'COMPLETED'].copy()
if len(measured):
    labels = [DATASET_TITLES[key].replace('Amazon ', '') for key in measured['dataset']]
    x = np.arange(len(measured))
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0), constrained_layout=True)
    axes[0].bar(x, measured['fit_seconds'] / 3600, color=COLORS['blue'])
    axes[0].set(ylabel='Coach.fit wall time (h)', title='Training + native evaluation / I/O')
    axes[1].bar(x, measured['preprocessing_seconds'], color=COLORS['green'])
    axes[1].set(ylabel='Preprocessing wall time (s)', title='Total model preprocessing')
    axes[2].bar(x - .18, measured['peak_training_allocated_mib'], width=.36, color=COLORS['blue'], label='Allocated')
    axes[2].bar(x + .18, measured['peak_training_reserved_mib'], width=.36, color=COLORS['orange'], label='Reserved')
    axes[2].set(ylabel='Training allocator peak (MiB)', title='Training-epoch CUDA memory')
    axes[2].legend(frameon=False)
    for ax in axes:
        ax.set_xticks(x, labels)
        ax.grid(axis='y', alpha=.2)
    fig.suptitle(f'ECLIPSE resource profile | seed {SEED}', fontsize=11)
    save_figure(fig, f'eclipse_wo_ec_seed{SEED}_resource_profile',
                'ECLIPSE resource measurements, not a comparison against another model. '
                'Coach.fit includes native evaluation and I/O; preprocessing is separate. '
                'Memory excludes evaluation peaks. Cache state and runtime remain in manifests.',
                [RUN_ROOT / 'tables' / f'eclipse_wo_ec_seed{SEED}_selected_test.csv'])


## 10. Aggregate actual w/o EC seeds only

Audited measured w/o EC seed-1 and seed-2 receipts are embedded with provenance. Completed current seed-3 runs are added after native receipt and contract checks. No need to upload previous checkpoints solely to aggregate.

Snapshots are checked before insertion; optional additional artifact roots are validated as in seed 2. Conflicting attempts stop aggregation; never choose by TEST score. Full-model reference metrics are protocol-only and never inserted. Missing seed-3 datasets remain NOT_MEASURED. Sample SD is reported only for n >= 2; no p-value or CI is manufactured from epochs.


In [ ]:
def collect_seed_results():
    records = {}
    def add(key, receipt, provenance):
        training_seed = receipt['seed']
        metrics = {str(name).upper(): value for name, value in receipt['metrics'].items()}
        for name in PRIMARY_METRICS:
            value = metrics.get(name)
            if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or not 0 <= value <= 1:
                raise ValueError(f'Invalid measured seed metric: {provenance}, {name}')
        row = {'method': 'ECLIPSE', 'arm': 'wo_ec', 'dataset': key, 'seed': training_seed,
               'selected_epoch': receipt['epoch'], **{name: metrics[name] for name in PRIMARY_METRICS},
               'graph_fingerprint': receipt['graph_fingerprint'], 'provenance': provenance}
        identity = (key, training_seed)
        if identity in records:
            prior = records[identity]
            if any(prior[field] != row[field] for field in ['selected_epoch', 'graph_fingerprint', *PRIMARY_METRICS]):
                raise ValueError(f'Conflicting attempts for {identity}: {prior["provenance"]} vs {provenance}. '
                                 'Resolve provenance; do not select the best TEST score.')
        else:
            records[identity] = row
    # BEGIN verified measured w/o EC seed-1 snapshots
    for key, measured in VERIFIED_WO_EC_SEED1.items():
        receipt = measured['receipt']
        audited = LEGACY_REUSE_AUDIT[key]
        if (measured['status'] != 'REUSED_VERIFIED' or receipt.get('arm') != 'wo_ec'
                or receipt.get('seed') != 1 or receipt.get('training_reused') is not True
                or receipt.get('epoch') != audited['selected_epoch']
                or receipt.get('checkpoint_sha256') != audited['files']['best_model.pt']['sha256']
                or receipt.get('source_hashes') != audited['source_hashes']
                or receipt.get('graph_fingerprint') != WO_EC_GRAPH[key]
                or receipt.get('runtime_deviations')):
            raise ValueError(f'Invalid embedded measured w/o EC seed-1 snapshot: {key}')
        for field in ('python', 'torch', 'freerec'):
            if receipt.get('evaluation_runtime', {}).get(field) != audited['runtime'][field]:
                raise ValueError(f'Embedded seed-1 reevaluation runtime mismatch: {key}, {field}')
        validate_receipt(key, receipt, audited['legacy_manifest'], 1)
        add(key, receipt, 'verified measured native seed-1 receipt: ' + measured['provenance']
            + ' | SHA256=' + measured['receipt_sha256'])
    # END verified measured w/o EC seed-1 snapshots
    # Audited measured seed-2 snapshots; these are not model initialization state.
    for key, measured in VERIFIED_WO_EC_SEED2.items():
        receipt = measured['receipt']
        manifest = measured['manifest_projection']
        if measured['status'] != 'VERIFIED_ARTIFACT' or receipt.get('seed') != 2:
            raise ValueError(f'Invalid measured w/o EC seed-2 identity: {key}')
        for field, value in [('receipt', receipt), ('manifest_projection', manifest)]:
            canonical = json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False)
            actual = hashlib.sha256(canonical.encode('utf-8')).hexdigest()
            if actual != measured[field + '_canonical_sha256']:
                raise ValueError(f'Changed embedded seed-2 snapshot: {key}, {field}')
        compare_manifest(key, manifest, 2)
        validate_receipt(key, receipt, manifest, 2)
        reference = SEED1_REFERENCE[key]
        if (manifest['commit'] != RUNTIME_COMMIT
                or manifest['checkpoint_contract']['sources'] != reference['contract']['sources']
                or any(manifest[field] != reference[field] for field in ('python', 'torch', 'freerec'))):
            raise ValueError(f'Embedded seed-2 protocol/source/runtime mismatch: {key}')
        add(key, receipt, 'audited measured native seed-2 receipt: ' + measured['provenance']
            + ' | SHA256=' + measured['receipt_sha256'])

    roots = [RUN_ROOT / 'artifacts', *[Path(path).expanduser() for path in ADDITIONAL_ARTIFACT_ROOTS]]
    seen_paths = set()
    for root in roots:
        if not root.exists():
            print('No artifact root:', root)
            continue
        candidates = [root] if root.is_file() and root.name == 'selected_metrics.json' else root.rglob('selected_metrics.json')
        for path in candidates:
            identity_path = str(path.resolve())
            if identity_path in seen_paths:
                continue
            seen_paths.add(identity_path)
            receipt = json.loads(path.read_text(encoding='utf-8'))
            if receipt.get('evidence_mode') != 'pooled_activity_excess' or receipt.get('evidence_delta') != 0.0:
                print('Excluded another arm:', path)
                continue
            if receipt.get('seed') not in (1, 2, 3):
                print('Excluded a seed outside the locked three-seed protocol:', path)
                continue
            key = next((key for key, reference in SEED1_REFERENCE.items()
                        if WO_EC_GRAPH[key] == receipt.get('graph_fingerprint')), None)
            if key is None:
                raise ValueError(f'Unknown w/o EC graph fingerprint: {path}')
            manifest_path, status_path = path.parent / 'manifest.json', path.parent / 'run_status.json'
            if not manifest_path.is_file() or not status_path.is_file():
                raise FileNotFoundError(f'Missing provenance beside {path}')
            status = json.loads(status_path.read_text(encoding='utf-8'))
            if status.get('status') != 'COMPLETED' or status.get('seed') != receipt['seed']:
                print('Excluded an incomplete attempt:', path)
                continue
            manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
            legacy_reuse = receipt.get('training_reused') is True
            if legacy_reuse:
                if receipt['seed'] != 1 or receipt.get('arm') != 'wo_ec':
                    raise ValueError(f'Invalid legacy reuse identity: {path}')
                contract = manifest.get('checkpoint_contract', {})
                reference = SEED1_REFERENCE[key]
                expected_record = LEGACY_REUSE_AUDIT[key]
                if manifest != expected_record['legacy_manifest']:
                    raise ValueError(f'Legacy manifest is not the audited attempt: {path}')
                for field in ('splits', 'candidate_hashes', 'data_fingerprint'):
                    if contract.get(field) != reference['contract'][field]:
                        raise ValueError(f'Legacy data/candidate mismatch: {path}, {field}')
                if contract.get('graph_fingerprint') != WO_EC_GRAPH[key]:
                    raise ValueError(f'Legacy operator mismatch: {path}')
                if receipt.get('checkpoint_sha256') != expected_record['files']['best_model.pt']['sha256']:
                    raise ValueError(f'Legacy checkpoint mismatch: {path}')
                checkpoint = path.parent / 'best_model.pt'
                if not checkpoint.is_file() or file_sha256(checkpoint) != receipt['checkpoint_sha256']:
                    raise ValueError(f'Missing or altered legacy checkpoint: {path}')
                if receipt.get('epoch') != expected_record['selected_epoch']:
                    raise ValueError(f'Legacy selected epoch mismatch: {path}')
                if receipt.get('source_hashes') != expected_record['source_hashes']:
                    raise ValueError(f'Legacy evaluation source mismatch: {path}')
                if receipt.get('runtime_deviations') or status.get('training_reused') is not True:
                    raise ValueError(f'Unmatched legacy reevaluation runtime: {path}')
                for field in ('torch', 'freerec'):
                    if receipt.get('evaluation_runtime', {}).get(field) != reference[field]:
                        raise ValueError(f'Legacy reevaluation runtime mismatch: {path}')
                if str(receipt.get('evaluation_runtime', {}).get('python', '')).split('.')[:2] != reference['python'].split('.')[:2]:
                    raise ValueError(f'Legacy reevaluation Python major/minor mismatch: {path}')
            else:
                compare_manifest(key, manifest, receipt['seed'])
            validate_receipt(key, receipt, manifest, receipt['seed'])
            reference = SEED1_REFERENCE[key]
            if any(manifest.get(field) != reference[field] for field in ('torch', 'freerec')):
                raise ValueError(f'Runtime mismatch; do not silently pool seeds: {path}')
            if str(manifest.get('python', '')).split('.')[:2] != reference['python'].split('.')[:2]:
                raise ValueError(f'Python major/minor mismatch: {path}')
            for name, expected in ({} if legacy_reuse else reference['contract']['sources']).items():
                actual = manifest.get('checkpoint_contract', {}).get('sources', {}).get(name)
                if actual != expected:
                    own_line_endings_only = (path.is_relative_to(RUN_ROOT)
                                             and source_report.get(name, {}).get('line_endings_only') is True
                                             and actual == source_report[name]['sha256'])
                    if not own_line_endings_only:
                        raise ValueError(f'Algorithm source mismatch: {path}, {name}')
            add(key, receipt, str(path))
    return pd.DataFrame(records.values(), columns=['method', 'arm', 'dataset', 'seed', 'selected_epoch', *PRIMARY_METRICS, 'graph_fingerprint', 'provenance']).sort_values(['dataset', 'seed']).reset_index(drop=True)


SEED_RESULTS = collect_seed_results()
SEED_RESULTS.to_csv(RUN_ROOT / 'tables' / 'eclipse_wo_ec_measured_seeds.csv', index=False, na_rep='NOT_MEASURED')
status_grid = pd.MultiIndex.from_product([['baby', 'sports', 'electronics'], [1, 2, 3]], names=['dataset', 'seed'])
status_table = SEED_RESULTS.set_index(['dataset', 'seed']).reindex(status_grid).reset_index()
status_table['measurement_status'] = np.where(status_table['NDCG@20'].notna(), 'MEASURED', 'NOT_MEASURED')
status_table.to_csv(RUN_ROOT / 'tables' / 'eclipse_wo_ec_seed_status.csv', index=False, na_rep='NOT_MEASURED')
display(status_table[['dataset', 'seed', 'measurement_status', *PRIMARY_METRICS]].fillna('NOT_MEASURED'))
aggregate_rows = []
for key, group in SEED_RESULTS.groupby('dataset', sort=False):
    count = len(group)
    for metric in PRIMARY_METRICS:
        aggregate_rows.append({'method': 'ECLIPSE', 'arm': 'wo_ec', 'dataset': key, 'metric': metric,
                               'n_seeds': count, 'seeds': ','.join(map(str, sorted(group['seed']))),
                               'mean': float(group[metric].mean()),
                               'sample_std': float(group[metric].std(ddof=1)) if count >= 2 else None,
                               'complete_three_seed_protocol': set(group['seed']) == {1, 2, 3}})
AGGREGATE = pd.DataFrame(aggregate_rows)
AGGREGATE.to_csv(RUN_ROOT / 'tables' / 'eclipse_wo_ec_mean_std.csv', index=False, na_rep='NOT_MEASURED')
display(AGGREGATE.fillna('NOT_MEASURED'))
write_json(RUN_ROOT / 'tables' / 'statistical_analysis_status.json', {
    'p_value': 'NOT_COMPUTED', 'reason': 'This notebook aggregates the w/o EC arm only; no matched full-model multi-seed sample is provided.',
    'seed_variability': 'sample SD of measured independent training seeds; n reported per dataset',
    'epoch_points_are_independent_samples': False,
    'per_user_testing': 'Not performed; requires paired native-evaluator per-user measurements.',
    'selection': 'validation NDCG@20; TEST at selected checkpoint only'})
latex_lines = [r'\begin{tabular}{lcrrrr}', r'\toprule',
               r'Dataset & Seeds & R@10 & R@20 & N@10 & N@20 \\', r'\midrule']
for key, group in SEED_RESULTS.groupby('dataset', sort=False):
    values = []
    for metric in PRIMARY_METRICS:
        mean = group[metric].mean()
        values.append(f'{mean:.6f}' if len(group) < 2 else f'${mean:.6f} \\pm {group[metric].std(ddof=1):.6f}$')
    latex_lines.append(f'{DATASET_TITLES[key]} & {len(group)} & ' + ' & '.join(values) + r' \\')
latex_lines += [r'\bottomrule', r'\end{tabular}', '% Uncertainty is sample SD, not a 95% confidence interval.']
(RUN_ROOT / 'tables' / 'eclipse_wo_ec_mean_std.tex').write_text('\n'.join(latex_lines) + '\n', encoding='utf-8')


## 11. Seed-stability figures

Mỗi điểm là một run thật. Mean ± SD chỉ tính từ seed đã đo; trục thể hiện vị trí seed thiếu nhưng không có điểm giả. Hình robustness này không thay thế ablation. Nếu chỉ có một seed thì không có SD/error bar.


In [ ]:
for metric, suffix in [('NDCG@20', 'ndcg20'), ('RECALL@20', 'recall20')]:
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.8), constrained_layout=True)
    for ax, key in zip(axes, ['baby', 'sports', 'electronics']):
        group = SEED_RESULTS.loc[SEED_RESULTS['dataset'] == key].sort_values('seed')
        ax.scatter(group['seed'], group[metric], color=COLORS['blue'], s=28, zorder=3)
        if len(group) >= 2:
            ax.errorbar([4], [group[metric].mean()], yerr=[group[metric].std(ddof=1)],
                        fmt='D', color=COLORS['orange'], capsize=4, label='Mean +/- sample SD')
            ax.legend(frameon=False, fontsize=7)
        ax.set_xticks([1, 2, 3, 4], ['1', '2', '3', f'Mean\n(n={len(group)})'])
        ax.set(xlim=(.6, 4.5), xlabel='Training seed', ylabel=f'Selected TEST {metric}', title=DATASET_TITLES[key])
        ax.grid(axis='y', alpha=.2)
        ax.ticklabel_format(axis='y', style='plain', useOffset=False)
    fig.suptitle(f'ECLIPSE w/o EC | observed seed variability in {metric}', fontsize=11)
    save_figure(fig, f'eclipse_wo_ec_seed_stability_{suffix}',
                'Blue points are measured independent training runs. Orange diamonds/error bars show '
                'mean and sample SD of available seeds (n shown), not confidence intervals. '
                'Missing seed positions have no point. Selection uses validation NDCG@20.',
                [RUN_ROOT / 'tables' / 'eclipse_wo_ec_measured_seeds.csv'])


## 12. Đóng gói artifact và dùng cho paper

Archive gồm YAML/CLI đã resolve, environment/pip freeze, source/data/comparability checks, logs, manifests, selected TEST receipts, telemetry, checkpoints, CSV, LaTeX, figure sources/captions. Không gói graph cache hoặc dữ liệu/features. Tải archive khỏi Kaggle trước khi session kết thúc.

**Paper:** architecture → component/evidence ablations (notebook riêng) → calibration diagnostics → efficiency comparison khi có dữ liệu comparator phù hợp. Learning/memory/seed-stability có thể dùng phụ lục. Không dùng bảng significance notebook cũ.

**Seed 3:** notebook này đã đặt `SEED = 3`; giữ nguyên tham số thuật toán. Seed 1/2 w/o EC đã được nhúng kèm provenance để tổng hợp, không dùng làm initialization. Chọn `GPU_INDEX` phù hợp; output tự tách theo seed/session. Không sửa YAML nguồn hoặc code thuật toán, không resume checkpoint seed khác.

Before batch execution, attach the processed datasets. The paper result is the selected-checkpoint TEST receipt, not any additional terminal TEST row in native logs.


In [ ]:
import zipfile

figure_entries = []
figure_manifest = RUN_ROOT / 'figures' / 'figure_manifest.json'
if figure_manifest.is_file():
    figure_entries = json.loads(figure_manifest.read_text(encoding='utf-8'))
caption_text = '# ECLIPSE figure captions and provenance\n\n'
for entry in figure_entries:
    caption_text += f'## {entry["stem"]}\n\n{entry["caption"]}\n\nSources:\n'
    caption_text += '\n'.join(f'- {path}' for path in entry['sources']) + '\n\n'
(RUN_ROOT / 'figures' / 'FIGURE_CAPTIONS.md').write_text(caption_text, encoding='utf-8')
readme = f"""# ECLIPSE w/o EC training artifacts
Training seed: {SEED}
Session: {SESSION_ID} (UTC identifier)
Algorithm reference commit: {RUNTIME_COMMIT}
Runtime checkout commit: {ACTUAL_COMMIT}
Selected datasets: {', '.join(SELECTED_DATASETS)}

Primary results: tables/eclipse_wo_ec_seed{SEED}_selected_test.csv
Native evidence: artifacts/*/selected_metrics.json and run_status.json
Seed status: tables/eclipse_wo_ec_seed_status.csv
Aggregate: tables/eclipse_wo_ec_mean_std.csv (sample SD; n reported)
Figures/captions: figures/*.pdf, *.png, FIGURE_CAPTIONS.md

Fit time is read from Coach.fit, not summed training epoch time.
Training allocator peaks exclude evaluation/driver/other-process memory.
No p-values from validation epochs or historical scalar baselines.
Full seed-1 metrics are protocol-only. Audited measured w/o EC seed-1 and seed-2 receipts are embedded with provenance. Missing seeds are NOT_MEASURED.
Seed-1/2 w/o EC measurements are embedded for aggregation only; current seed 3 is measured by this run.
"""
(RUN_ROOT / 'README.md').write_text(readme, encoding='utf-8')
archive = WORK_DIR / f'{RUN_ROOT.name}.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as handle:
    for path in sorted(RUN_ROOT.rglob('*')):
        if path.is_file():
            handle.write(path, arcname=str(Path(RUN_ROOT.name) / path.relative_to(RUN_ROOT)))
print(f'Archive: {archive}\nSize: {archive.stat().st_size / 2**20:.1f} MiB\nSHA256: {file_sha256(archive)}')
try:
    from IPython.display import FileLink
    display(FileLink(str(archive)))
except ImportError:
    pass
